# Line plot generator

In [ ]:
import json
import math
import re
import shutil
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import altair as alt
import kaleido
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
except Exception:
    sns = None

import plotly.graph_objects as go
import vl_convert

from groq import Groq
import os

# UNCOMMENT FOR OLLAMA
import ollama

In [ ]:
from pathlib import Path

# Project paths
PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent

DATA_TRAIN = Path(r"C:\\Users\\Michelle\\I2R\\data\\train")

# Normal generated outputs
#LINE_OUT_ROOT = PROJECT / "outputs" / "generated" / "lineplots"

LINE_OUT_ROOT = Path(r"C:\Users\Michelle\OneDrive - TU Eindhoven\I2R_dataset\lineplots")

# Testing outputs
LINE_TESTING_ROOT = PROJECT / "testing" / "line_testing"

LINE_OUT_ROOT.mkdir(parents=True, exist_ok=True)
LINE_TESTING_ROOT.mkdir(parents=True, exist_ok=True)

CLEAR_OUTPUT = True
LIBRARIES = ["altair", "matplotlib", "seaborn", "plotly"]

# Use metadata for consistency
SUBDIRS = ["images", "tables", "metadata"]

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

# Data

In [26]:
# ── Dataset registry ──────────────────────────────────────────────────────────
# Add new datasets here. Each entry needs:
#   "path"        : Path to the CSV file
#   "numeric_cols": List of numeric columns to use as y-axis values
#   "group_cols"  : List of categorical columns to use for series grouping
#   "date_col"    : Optional date column name (None if not applicable)
#   "loader"      : Optional function name to pre-process the dataframe (or None)
DATASET_REGISTRY = {
    "warehouse_retail": {
        "path": DATA_TRAIN / "Warehouse_and_Retail_Sales.csv",
        "numeric_cols": ["RETAIL SALES", "WAREHOUSE SALES", "RETAIL TRANSFERS"],
        "group_cols":   ["ITEM TYPE", "SUPPLIER"],
        "date_col":     "date",
        "agg_cols":     ["YEAR", "MONTH"],
        "loader":       "load_warehouse_retail",
    },
    # ── Add new datasets below ────────────────────────────────────────────────
    # "my_dataset": {
    #     "path":         DATA_TRAIN / "my_file.csv",
    #     "numeric_cols": ["col_a", "col_b"],
    #     "group_cols":   ["category"],
    #     "date_col":     None,
    #     "agg_cols":     None,
    #     "loader":       None,
    # },

    "london_borough_sector_jobs": {
        "path": DATA_TRAIN / "london_borough_sector_jobs.csv",
        "numeric_cols": ["EMPLOYEE_JOBS"],
        "group_cols":   ["SECTOR", "BOROUGH"],
        "date_col":     None,
        "agg_cols":     ["YEAR"],
        "loader":       "load_london_borough_sector_jobs",
    },    
}

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

In [27]:
# ── Dataset loaders ───────────────────────────────────────────────────────────
# Add a loader function for each dataset that needs pre-processing.
# The function receives the raw DataFrame and returns a cleaned one.

def load_warehouse_retail(df: pd.DataFrame) -> pd.DataFrame:
    df["date"] = pd.to_datetime(
        dict(year=df["YEAR"].astype("Int64"), month=df["MONTH"].astype("Int64"), day=1),
        errors="coerce",
    )
    return df

def load_london_borough_sector_jobs(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean London Datastore borough-by-sector employee jobs data.

    Handles the CSV format where the first rows are title/metadata rows and
    the real header row contains columns such as borough, sector, 1971, 1972, ...
    """

    import re

    df = df.copy()

    # Drop columns and rows that are completely empty
    df = df.dropna(axis=1, how="all")
    df = df.dropna(axis=0, how="all").reset_index(drop=True)

    def is_year_like(value) -> bool:
        s = str(value).strip()
        s = re.sub(r"\.0$", "", s)
        return bool(re.fullmatch(r"(19|20)\d{2}", s))

    def normalise_year(value) -> str | None:
        s = str(value).strip()
        s = re.sub(r"\.0$", "", s)
        if is_year_like(s):
            return s
        return None

    # ------------------------------------------------------------------
    # 1. If pandas read the wrong header, find the real header row.
    #    The real header row should contain lots of year columns.
    # ------------------------------------------------------------------
    current_year_cols = [c for c in df.columns if is_year_like(c)]

    if len(current_year_cols) < 5:
        header_row_idx = None
        best_year_count = 0

        for i in range(min(len(df), 30)):
            row_values = df.iloc[i].tolist()
            year_count = sum(is_year_like(v) for v in row_values)

            if year_count > best_year_count:
                best_year_count = year_count
                header_row_idx = i

        if header_row_idx is None or best_year_count < 5:
            raise ValueError(
                "Could not find the real header row containing year columns. "
                f"Columns found: {list(df.columns)}"
            )

        new_columns = df.iloc[header_row_idx].tolist()
        df = df.iloc[header_row_idx + 1:].copy()
        df.columns = new_columns

    # ------------------------------------------------------------------
    # 2. Clean column names.
    # ------------------------------------------------------------------
    cleaned_cols = []
    for c in df.columns:
        if pd.isna(c):
            cleaned_cols.append("")
        else:
            cleaned_cols.append(str(c).strip())

    df.columns = cleaned_cols
    df = df.dropna(axis=1, how="all")
    df = df.dropna(axis=0, how="all").reset_index(drop=True)

    # Normalise year column names
    renamed = {}
    for c in df.columns:
        y = normalise_year(c)
        if y is not None:
            renamed[c] = y

    df = df.rename(columns=renamed)

    # ------------------------------------------------------------------
    # 3. Identify year columns and descriptor columns.
    # ------------------------------------------------------------------
    year_cols = [c for c in df.columns if is_year_like(c)]

    if not year_cols:
        raise ValueError(
            "Could not find year columns after cleaning. "
            f"Columns found: {list(df.columns)}"
        )

    col_lookup = {str(c).lower().strip(): c for c in df.columns}

    def find_col(possible_names):
        for name in possible_names:
            key = name.lower().strip()
            if key in col_lookup:
                return col_lookup[key]
        return None

    borough_col = find_col([
        "borough",
        "borough name",
        "local authority",
        "local authority name",
        "area",
        "geography",
        "name",
    ])

    sector_col = find_col([
        "sector",
        "industry",
        "industry sector",
        "sector name",
        "sic section",
        "sic",
    ])

    # Fallback: infer from non-year columns if exact names are messy
    non_year_cols = [c for c in df.columns if c not in year_cols and str(c).strip() != ""]

    if borough_col is None or sector_col is None:
        # Usually these files have a few descriptor columns before the year columns.
        # We choose text-heavy columns with many unique values.
        candidate_cols = []

        for c in non_year_cols:
            non_null = df[c].dropna().astype(str).str.strip()
            if len(non_null) == 0:
                continue

            unique_count = non_null.nunique()
            avg_len = non_null.str.len().mean()

            candidate_cols.append((c, unique_count, avg_len))

        # Prefer columns with meaningful text values
        candidate_cols = sorted(candidate_cols, key=lambda x: (x[1], x[2]), reverse=True)
        inferred_cols = [c for c, _, _ in candidate_cols]

        if borough_col is None and len(inferred_cols) >= 1:
            borough_col = inferred_cols[0]

        if sector_col is None and len(inferred_cols) >= 2:
            sector_col = inferred_cols[1]

    if borough_col is None or sector_col is None:
        raise ValueError(
            "Could not identify borough and sector columns. "
            f"Columns found: {list(df.columns)}"
        )

    # ------------------------------------------------------------------
    # 4. Convert from wide format to long format.
    # ------------------------------------------------------------------
    df = df.melt(
        id_vars=[borough_col, sector_col],
        value_vars=year_cols,
        var_name="YEAR",
        value_name="EMPLOYEE_JOBS",
    )

    df = df.rename(columns={
        borough_col: "BOROUGH",
        sector_col: "SECTOR",
    })

    df["YEAR"] = pd.to_numeric(df["YEAR"], errors="coerce").astype("Int64")

    # Remove commas from numbers such as "12,300"
    df["EMPLOYEE_JOBS"] = (
        df["EMPLOYEE_JOBS"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    df["EMPLOYEE_JOBS"] = pd.to_numeric(df["EMPLOYEE_JOBS"], errors="coerce")

    df["BOROUGH"] = df["BOROUGH"].astype(str).str.strip()
    df["SECTOR"] = df["SECTOR"].astype(str).str.strip()

    df = df.dropna(subset=["YEAR", "BOROUGH", "SECTOR", "EMPLOYEE_JOBS"])

    # Remove empty / total rows that are not useful for multi-line charts
    df = df[
        ~df["BOROUGH"].str.lower().isin([
            "nan",
            "",
            "total",
            "london",
            "greater london",
        ])
    ]

    df = df[
        ~df["SECTOR"].str.lower().isin([
            "nan",
            "",
            "total",
            "all",
            "all industries",
            "all sectors",
        ])
    ]

    return df.reset_index(drop=True)


def get_loader(name: str):
    loaders = {
        "load_warehouse_retail": load_warehouse_retail,
        "load_london_borough_sector_jobs": load_london_borough_sector_jobs,
    }
    return loaders.get(name)


def load_dataset(dataset_name: str) -> pd.DataFrame | None:
    """Load and pre-process a registered dataset. Returns None if file not found."""
    spec = DATASET_REGISTRY.get(dataset_name)
    if spec is None:
        raise KeyError(f"Unknown dataset: {dataset_name}")
    path = spec["path"]
    if not path.exists():
        print(f"Dataset not found: {path}")
        return None
    df = pd.read_csv(path)
    if spec["loader"]:
        loader_fn = get_loader(spec["loader"])
        if loader_fn:
            df = loader_fn(df)
    return df


# Load all available datasets
DATASETS = {}
for ds_name, ds_spec in DATASET_REGISTRY.items():
    ds = load_dataset(ds_name)
    if ds is not None:
        DATASETS[ds_name] = ds
        print(f"Loaded '{ds_name}': {len(ds)} rows, cols: {list(ds.columns)}")

# Pick default dataset (first available)
DEFAULT_DATASET = next(iter(DATASETS)) if DATASETS else None
print(f"Default dataset: {DEFAULT_DATASET}")

Loaded 'warehouse_retail': 307645 rows, cols: ['YEAR', 'MONTH', 'SUPPLIER', 'ITEM CODE', 'ITEM DESCRIPTION', 'ITEM TYPE', 'RETAIL SALES', 'RETAIL TRANSFERS', 'WAREHOUSE SALES', 'date']
Loaded 'london_borough_sector_jobs': 28508 rows, cols: ['BOROUGH', 'SECTOR', 'YEAR', 'EMPLOYEE_JOBS']
Default dataset: warehouse_retail


In [28]:
# def sample_line_data_from_df(
#     df: pd.DataFrame,
#     spec: dict,
#     rng: np.random.Generator,
#     style: dict,
#     min_points: int = 3,
# ) -> tuple[pd.DataFrame, dict] | None:
#     """Sample a line-plottable DataFrame from a real dataset."""
#     numeric_cols = spec["numeric_cols"]
#     agg_cols     = spec.get("agg_cols")
#     n_lines      = max(1, int(style.get("n_lines", 1)))

#     if not numeric_cols or not agg_cols:
#         return None

#     y_col = str(rng.choice(numeric_cols))

#     strategies = ["time"]
#     if spec.get("group_cols"):
#         strategies.append("group_series")
#     strategy = str(rng.choice(strategies))

#     # Build global sorted period list — shared x-axis for all strategies
#     all_periods = (
#         df.groupby(agg_cols)[numeric_cols[0]].sum()
#         .reset_index()[agg_cols]
#         .drop_duplicates()
#         .sort_values(agg_cols)
#         .reset_index(drop=True)
#     )
#     all_periods["x"]       = np.arange(len(all_periods), dtype=float)
#     all_periods["x_label"] = (all_periods[agg_cols[0]].astype(str) + "-"
#                                + all_periods[agg_cols[1]].astype(str).str.zfill(2))
#     period_to_x      = {(r[agg_cols[0]], r[agg_cols[1]]): r["x"] for _, r in all_periods.iterrows()}
#     x_tick_positions = all_periods["x"].tolist()
#     x_tick_labels    = all_periods["x_label"].tolist()

#     if strategy == "time":
#         plot_df = (
#             df.groupby(agg_cols)[[y_col]].sum()
#             .reset_index()
#             .sort_values(agg_cols)
#             .reset_index(drop=True)
#         )
#         plot_df["x"]       = plot_df.apply(
#             lambda r: period_to_x[(r[agg_cols[0]], r[agg_cols[1]])], axis=1)
#         plot_df["x_label"] = (plot_df[agg_cols[0]].astype(str) + "-"
#                                + plot_df[agg_cols[1]].astype(str).str.zfill(2))
#         plot_df["series"]  = "Series 1"

#     else:  # group_series
#         gc = str(rng.choice(spec["group_cols"]))
#         if gc not in df.columns:
#             return None
#         top_cats = (
#             df.dropna(subset=[gc])
#             .groupby(gc)[y_col].sum()
#             .nlargest(n_lines).index.tolist()
#         )
#         if not top_cats:
#             return None
#         rows = []
#         for cat in top_cats:
#             sub = (
#                 df[df[gc] == cat]
#                 .groupby(agg_cols)[[y_col]].sum()
#                 .reset_index()
#                 .sort_values(agg_cols)
#                 .reset_index(drop=True)
#             )
#             sub["x"]       = sub.apply(
#                 lambda r: period_to_x.get((r[agg_cols[0]], r[agg_cols[1]]), np.nan), axis=1)
#             sub["x_label"] = (sub[agg_cols[0]].astype(str) + "-"
#                                + sub[agg_cols[1]].astype(str).str.zfill(2))
#             sub["series"]  = str(cat)
#             rows.append(sub)
#         plot_df = pd.concat(rows, ignore_index=True)

#     plot_df = plot_df.dropna(subset=[y_col, "x"]).rename(columns={y_col: "y"})

#     if len(plot_df) < min_points:
#         return None

#     context = {
#         "x_label":                "Time period",
#         "y_label":                y_col.title(),
#         "n_lines":                int(plot_df["series"].nunique()),
#         "x_tick_positions":       x_tick_positions,
#         "x_tick_labels":          x_tick_labels,
#         "x_axis_month_day_labels": 0,
#         "strategy":               strategy,
#     }
#     return plot_df.reset_index(drop=True), context

In [29]:
def sample_line_data_from_df(
    df: pd.DataFrame,
    spec: dict,
    rng: np.random.Generator,
    style: dict,
    min_points: int = 3,
) -> tuple[pd.DataFrame, dict] | None:
    """Sample a line-plottable DataFrame from a real dataset."""
    numeric_cols = spec["numeric_cols"]
    agg_cols     = spec.get("agg_cols")
    n_lines      = max(1, int(style.get("n_lines", 1)))

    if not numeric_cols or not agg_cols:
        return None

    y_col = str(rng.choice(numeric_cols))

    available_numeric = [c for c in numeric_cols if c in df.columns]
    if not available_numeric:
        return None

    if y_col not in df.columns:
        y_col = str(rng.choice(available_numeric))

    available_agg_cols = [c for c in agg_cols if c in df.columns]
    if len(available_agg_cols) != len(agg_cols):
        return None

    agg_cols = available_agg_cols

    strategies = ["time"]
    if spec.get("group_cols"):
        strategies.append("group_series")

    strategy = str(rng.choice(strategies))

    def make_period_label(frame: pd.DataFrame) -> pd.Series:
        if len(agg_cols) == 1:
            return frame[agg_cols[0]].astype(str)

        labels = frame[agg_cols[0]].astype(str)
        for col in agg_cols[1:]:
            if col.upper() == "MONTH":
                labels = labels + "-" + frame[col].astype(str).str.zfill(2)
            else:
                labels = labels + "-" + frame[col].astype(str)
        return labels

    def period_key_from_row(row):
        if len(agg_cols) == 1:
            return row[agg_cols[0]]
        return tuple(row[col] for col in agg_cols)

    # Build global sorted period list — shared x-axis
    all_periods = (
        df.groupby(agg_cols)[y_col].sum()
        .reset_index()[agg_cols]
        .drop_duplicates()
        .sort_values(agg_cols)
        .reset_index(drop=True)
    )

    all_periods["x"] = np.arange(len(all_periods), dtype=float)
    all_periods["x_label"] = make_period_label(all_periods)

    if len(agg_cols) == 1:
        period_to_x = {
            r[agg_cols[0]]: r["x"]
            for _, r in all_periods.iterrows()
        }
    else:
        period_to_x = {
            tuple(r[col] for col in agg_cols): r["x"]
            for _, r in all_periods.iterrows()
        }

    x_tick_positions = all_periods["x"].tolist()
    x_tick_labels    = all_periods["x_label"].tolist()

    if strategy == "time":
        plot_df = (
            df.groupby(agg_cols)[[y_col]].sum()
            .reset_index()
            .sort_values(agg_cols)
            .reset_index(drop=True)
        )

        plot_df["x"] = plot_df.apply(
            lambda r: period_to_x.get(period_key_from_row(r), np.nan),
            axis=1,
        )
        plot_df["x_label"] = make_period_label(plot_df)
        plot_df["series"] = "Series 1"

    else:
        group_cols = [gc for gc in spec["group_cols"] if gc in df.columns]
        if not group_cols:
            return None

        gc = str(rng.choice(group_cols))

        top_cats = (
            df.dropna(subset=[gc])
            .groupby(gc)[y_col].sum()
            .nlargest(n_lines)
            .index
            .tolist()
        )

        if not top_cats:
            return None

        rows = []

        for cat in top_cats:
            sub = (
                df[df[gc] == cat]
                .groupby(agg_cols)[[y_col]].sum()
                .reset_index()
                .sort_values(agg_cols)
                .reset_index(drop=True)
            )

            sub["x"] = sub.apply(
                lambda r: period_to_x.get(period_key_from_row(r), np.nan),
                axis=1,
            )
            sub["x_label"] = make_period_label(sub)
            sub["series"] = str(cat)

            rows.append(sub)

        plot_df = pd.concat(rows, ignore_index=True)

    plot_df = plot_df.dropna(subset=[y_col, "x"]).rename(columns={y_col: "y"})

    if len(plot_df) < min_points:
        return None

    context = {
        "x_label": "Time period",
        "y_label": y_col.replace("_", " ").title(),
        "n_lines": int(plot_df["series"].nunique()),
        "x_tick_positions": x_tick_positions,
        "x_tick_labels": x_tick_labels,
        "x_axis_month_day_labels": 0,
        "strategy": strategy,
        "source_agg_cols": agg_cols,
    }

    return plot_df.reset_index(drop=True), context

## 1. Sampling weights

Weights are derived directly from observed frequencies in the reference dataset.
Each parameter maps `numeric_code → probability`. No name-mapping layer — renderers interpret codes directly.

In [30]:
# Sampling weights derived from observed dataset frequencies.
# Each key is a parameter name; values are {numeric_code: probability}.
# Codes match the column coding scheme in the reference Excel file.
# Updated from linecharts_with_fractions.xlsx / Fractions sheet.
SAMPLING_WEIGHTS = {
    "title_present": {
        0: 0.32,
        1: 0.68,
    },
    "title_location": {
        0: 0.36,
        1: 0.49,
        2: 0.14,
        3: 0.01,
    },
    "title_color": {
        0: 0.64,
        1: 0.01,
        2: 0.01,
        3: 0.13,
        4: 0.07,
        6: 0.11,
        7: 0.03,
    },
    "title_size": {
        0: 0.58,
        1: 0.22,
        2: 0.13,
        3: 0.07,
    },
    "subtitle_present": {
        0: 0.92,
        1: 0.08,
    },
    "legend_present": {
        0: 0.56,
        1: 0.44,
    },
    "legend_title_size": {
        0: 0.73,
        1: 0.27,
    },
    "legend_title_color": {
        0: 0.95,
        1: 0.05,
    },
    "legend_text_color": {
        0: 0.59,
        1: 0.06,
        2: 0.16,
        3: 0.10,
        4: 0.06,
        5: 0.02,
        6: 0.01,
    },
    "legend_outline": {
        0: 0.94,
        1: 0.06,
    },
    "legend_fill": {
        0: 0.55,
        2: 0.39,
        3: 0.04,
        4: 0.02,
    },
    "legend_orientation": {
        0: 0.56,
        2: 0.05,
        3: 0.01,
        4: 0.21,
        6: 0.04,
        7: 0.02,
        8: 0.11,
    },
    "direct_labels": {
        0: 0.76,
        1: 0.23,
        2: 0.01,
    },
    "label_content": {
        0: 0.78,
        2: 0.22,
    },
    "label_color": {
        0: 0.78,
        2: 0.14,
        3: 0.05,
        4: 0.03,
    },
    "chart_outline": {
        0: 0.09,
        1: 0.39,
        2: 0.23,
        4: 0.29,
    },
    "gridlines": {
        0: 0.20,
        2: 0.45,
        3: 0.30,
        4: 0.05,
    },
    "gridline_color": {
        0: 0.23,
        1: 0.07,
        2: 0.12,
        3: 0.43,
        4: 0.02,
        5: 0.03,
        6: 0.02,
        7: 0.04,
        8: 0.04,
    },
    "image_outline": {
        0: 0.8108108108,
        1: 0.1891891892,
    },
    "background": {
        0: 0.03,
        1: 0.83,
        2: 0.02,
        3: 0.04,
        4: 0.08,
    },
    "axis_text_orientation": {
        0: 0.40,
        1: 0.02,
        2: 0.52,
        3: 0.01,
        4: 0.02,
        5: 0.02,
        6: 0.01,
    },
    "axis_text_color": {
        0: 0.60,
        1: 0.01,
        3: 0.07,
        4: 0.09,
        5: 0.19,
        6: 0.04,
    },
    "axis_color": {
        0: 0.40,
        1: 0.19,
        2: 0.32,
        3: 0.06,
        4: 0.03,
    },
    "line_structure": {
        0: 0.82,
        1: 0.18,
    },
    "line_pattern": {
        0: 0.98,
        3: 0.02,
    },
    "n_lines": {
        1: 0.50,
        2: 0.15,
        3: 0.20,
        4: 0.11,
        5: 0.02,
        8: 0.01,
        15: 0.01,
    },
    "line_start": {
        0: 0.51,
        1: 0.49,
    },
    "point_shape_mode": {
        0: 0.53,
        1: 0.02,
        2: 0.06,
        3: 0.38,
        4: 0.01,
    },
    "point_same_color": {
        0: 0.05,
        1: 0.48,
        2: 0.04,
        3: 0.37,
        4: 0.03,
        5: 0.03,
    },
    "palette_type": {
        0: 0.04,
        1: 0.03,
        2: 0.29,
        3: 0.05,
        4: 0.15,
        5: 0.10,
        6: 0.01,
        7: 0.09,
        8: 0.02,
        9: 0.02,
        10: 0.03,
        11: 0.03,
        13: 0.07,
        14: 0.02,
        15: 0.02,
        16: 0.01,
        17: 0.01,
        18: 0.01,
    },

    # No exact matching column name in the Excel sheet, so this remains from the earlier notebook.
    "x_axis_month_day_labels": {
        0: 0.75,
        1: 0.12,
        2: 0.08,
        3: 0.05,
    },

    # From "Jumps between X axes values".
    "x_major_ticks": {
        0: 0.67,
        1: 0.33,
    },

    # From "Minor ticks x axes (yes)".
    "x_minor_ticks": {
        0: 0.89,
        1: 0.11,
    },

    "y_major_ticks": {
        0: 0.69,
        1: 0.31,
    },
    # From "Minor ticks y axes (yes)".
    "y_minor_ticks": {
        0: 0.87,
        1: 0.13,
    },
}


In [31]:
# Optional: recompute sampling weights directly from an updated reference Excel file.
# This mapping avoids accidental column shifts when the Excel sheet contains columns
# that are not used by the generator, such as Scale X/Y axes or Line orientation.
EXCEL_WEIGHT_COLUMNS = {
    "title_present": "Title present",
    "title_location": "Title location",
    "title_color": "Title color ",
    "title_size": "Title size ",
    "subtitle_present": "Subtitle present? ",
    "legend_present": "Legend present ",
    "legend_title_size": "Legend title size",
    "legend_title_color": "Legend title color (yes) ",
    "legend_text_color": "Legend text color",
    "legend_outline": "Legend outline ",
    "legend_fill": "Legend different color",
    "legend_orientation": "Legend orientation",
    "direct_labels": "Direct label(s) on points on the line",
    "label_content": "content of Labels on point on line ",
    "label_color": "Color labels",
    "chart_outline": "Full outline of chart ",
    "gridlines": "Gridlines",
    "gridline_color": "gridlines color",
    "image_outline": "Outline of full image ",
    "background": "Background color",
    "axis_text_orientation": "Orientation of text x and y axes",
    "axis_text_color": "Color of text x and y axes",
    "axis_color": "Color of axes",
    "line_structure": "Line structure",
    "line_pattern": "Line pattern",
    "n_lines": "Number of lines in lineplot",
    "line_start": "Start of line ",
    "point_shape_mode": "shape of datapoints visible on the line",
    "point_same_color": "color of datapoint is same as color of line",
    "palette_type": "Palette type lines",
    # x_axis_month_day_labels has no direct observed column in the coding sheet.
    "x_major_ticks": "Jumps between X axes values",
    "x_minor_ticks": "Minor ticks x axes (yes)",
    "y_major_ticks": "Jumps between y-axes values",
    "y_minor_ticks": "Minor ticks y axes (yes)",
}


def load_weights_from_excel(xlsx_path: Path, sheet: str = "ToFill_line") -> dict:
    """Recompute sampling weights live from the reference Excel file.

    The source file alternates coded rows with URL/source rows. This function
    counts only rows whose first column matches line_XX and returns a dictionary
    matching the structure of SAMPLING_WEIGHTS.
    """
    df = pd.read_excel(xlsx_path, sheet_name=sheet, header=0)
    data = df[df.iloc[:, 0].astype(str).str.match(r"line_\d+", na=False)]

    weights = {}
    for param, excel_col in EXCEL_WEIGHT_COLUMNS.items():
        if excel_col not in data.columns:
            print(f"Skipping {param!r}: column {excel_col!r} not found")
            continue

        vals = pd.to_numeric(data[excel_col], errors="coerce").dropna()
        counts = vals.value_counts().sort_index()
        total = len(vals)
        if total > 0:
            weights[param] = {
                int(k): round(v / total, 10)
                for k, v in counts.items()
            }

    # Preserve this manually defined generator parameter if not in the Excel sheet.
    if "x_axis_month_day_labels" in SAMPLING_WEIGHTS:
        weights["x_axis_month_day_labels"] = SAMPLING_WEIGHTS["x_axis_month_day_labels"]

    return weights


def normalize_weights(weights: dict) -> dict:
    """Ensure each parameter's weights sum to 1.0."""
    out = {}
    for param, codes in weights.items():
        total = float(sum(codes.values()))
        if total <= 0:
            raise ValueError(f"Weights for {param!r} sum to zero")
        out[param] = {int(k): float(v) / total for k, v in codes.items()}
    return out


def sample_style(rng: np.random.Generator, weights: dict) -> dict:
    style = {}
    for param, code_probs in weights.items():
        codes = list(code_probs.keys())
        probs = np.array(list(code_probs.values()), dtype=float)
        probs /= probs.sum()
        style[param] = int(rng.choice(codes, p=probs))
    return style


### Validate sampling weights

This cell normalizes the observed fractions before generation. It also displays a quick check so small rounding differences do not affect sampling.


In [32]:
# Validate and normalize observed sampling weights.
# Use OBSERVED_WEIGHTS in generation cells; it is normalized to handle tiny rounding differences.
OBSERVED_WEIGHTS = normalize_weights(SAMPLING_WEIGHTS)

weight_check = pd.DataFrame([
    {
        "parameter": param,
        "n_codes": len(codes),
        "raw_sum": round(sum(codes.values()), 10),
        "normalized_sum": round(sum(OBSERVED_WEIGHTS[param].values()), 10),
    }
    for param, codes in SAMPLING_WEIGHTS.items()
])

weight_check


,parameter,n_codes,raw_sum,normalized_sum
0,title_present,2,1.0,1.0
1,title_location,4,1.0,1.0
2,title_color,7,1.0,1.0
3,title_size,4,1.0,1.0
4,subtitle_present,2,1.0,1.0
5,legend_present,2,1.0,1.0
6,legend_title_size,2,1.0,1.0
7,legend_title_color,2,1.0,1.0
8,legend_text_color,7,1.0,1.0
9,legend_outline,2,1.0,1.0


## 2. Synthetic line data sampling

In [33]:
# ── Hard-coded axis scale / range / orientation ───────────────────────────────
# These are fixed values used for synthetic data generation.
# Change them here to adjust all synthetically generated charts globally.

X_SCALE          = "0_20"         # fixed x-axis scale
Y_SCALE          = "0_100"        # fixed y-axis scale
LINE_ORIENTATION = "ascending"    # fixed line orientation


def get_x_values(style: dict):
    """Return (positions, axis_label, tick_labels) for the current X_SCALE / month/day setting."""
    mode = int(style.get("x_axis_month_day_labels", 0))
    if mode == 1:
        labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                  "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
        return np.arange(len(labels)), "Month", labels
    if mode == 2:
        labels = ["Mon", "Tues", "Wed", "Thurs", "Fri", "Sat", "Sun"]
        return np.arange(len(labels)), "Day", labels
    if mode == 3:
        labels = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
        return np.arange(len(labels)), "Day", labels
    # Fixed numeric scale
    x = np.arange(0, 21, 2)
    return x, "Measurement step", [str(v) for v in x]


def get_y_limits():
    """Return (y_min, y_max) for the current Y_SCALE."""
    return {
        "0_50":          (0, 50),
        "0_100":         (0, 100),
        "0_1000":        (0, 1000),
        "0_10":          (0, 10),
        "0_200":         (0, 200),
        "0_5":           (0, 5),
        "0_100_percent": (0, 100),
        "2400_2900":     (2400, 2900),
        "0_2000":        (0, 2000),
        "0_30":          (0, 30),
    }.get(Y_SCALE, (0, 100))


def make_base_line(x: np.ndarray, y_min: float, y_max: float,
                   rng: np.random.Generator) -> np.ndarray:
    """Generate a single line using the fixed LINE_ORIENTATION."""
    n    = len(x)
    span = y_max - y_min
    low  = y_min + span * 0.18
    high = y_min + span * 0.82
    if LINE_ORIENTATION == "ascending":
        y = np.linspace(low, high, n)
    elif LINE_ORIENTATION == "descending":
        y = np.linspace(high, low, n)
    elif LINE_ORIENTATION == "mostly_ascending":
        y = np.linspace(low, high, n) + rng.normal(0, span * 0.07, n)
    elif LINE_ORIENTATION == "mostly_descending":
        y = np.linspace(high, low, n) + rng.normal(0, span * 0.07, n)
    else:   # up_down
        y = y_min + span * (0.5 + 0.25 * np.sin(np.linspace(0, 2.5 * np.pi, n)))
    y += rng.normal(0, span * 0.035, n)
    return np.clip(y, y_min, y_max)


def sample_line_data(rng: np.random.Generator, style: dict) -> tuple:
    """Return (df, context) for the current style using synthetic data."""
    x, x_label, x_tick_labels = get_x_values(style)
    y_min, y_max = get_y_limits()
    n_lines = max(1, int(style.get("n_lines", 1)))

    if int(style.get("line_start", 0)) == 1 and len(x) > 1 and x[0] == 0:
        x = x[1:]
        x_tick_labels = x_tick_labels[1:]

    rows = []
    for i in range(n_lines):
        y = make_base_line(x, y_min, y_max, rng)
        offset = (i - (n_lines - 1) / 2) * (y_max - y_min) * 0.06
        y = np.clip(y + offset, y_min, y_max)
        for j, (xv, yv) in enumerate(zip(x, y)):
            rows.append({"x": float(xv), "x_label": x_tick_labels[j],
                         "y": float(yv), "series": f"Line {i + 1}"})

    df = pd.DataFrame(rows)
    context = {
        "x_label":               x_label,
        "y_label":               "Value (%)" if Y_SCALE == "0_100_percent" else "Value",
        "n_lines":               n_lines,
        "x_tick_positions":      list(map(float, x)),
        "x_tick_labels":         list(x_tick_labels),
        "x_axis_month_day_labels": int(style.get("x_axis_month_day_labels", 0)),
    }
    return df, context

## 3. Shared styling helpers

In [34]:
# ── Colour lookup tables (numeric code → value) ───────────────────────────────

BACKGROUNDS = {
    0: "transparent",
    1: "white",
    2: "light_gray",
    3: "dark_gray",
    4: "light_color",
}
BACKGROUND_COLOR_BASES = {
    0: None,                  # transparent
    1: (255, 255, 255),       # white
    2: (242, 242, 242),       # light gray
    3: (77,  77,  77),        # dark gray
    4: (237, 244, 255),       # light blue
}

TITLE_COLOR_BASES = {
    0: (0, 0, 0),
    1: (255, 127, 0),
    2: (200, 30, 30),
    3: (100, 100, 100),
    4: (0, 0, 0),
    5: (30, 150, 30),
    6: (255, 255, 255),
}
TITLE_SIZE_BASES = {0: 16, 1: 12, 2: 20, 3: 16}

TITLE_LOCATIONS = {
    0: ("none",   0.50, "center", "middle"),
    1: ("center", 0.50, "center", "middle"),
    2: ("left",   0.01, "left",   "start"),
    3: ("right",  0.99, "right",  "end"),
}

LEGEND_ORIENTATIONS = {
    0: None,
    1: ("left",      "center left",  (-0.22, 0.5),  dict(x=-0.22, y=0.5,  xanchor="right",  yanchor="middle")),
    2: ("right",     "center left",  (1.02,  0.5),  dict(x=1.02,  y=0.5,  xanchor="left",   yanchor="middle")),
    3: ("top",       "lower center", (0.5,   1.04), dict(x=0.5,   y=1.02, xanchor="center", yanchor="bottom")),
    4: ("bottom", "upper center", (0.5, -0.12), dict(x=0.5, y=-0.22, xanchor="center", yanchor="top")),
    5: ("top-left",  "lower left",   (0.0,   1.02), dict(x=0.0,   y=1.02, xanchor="left",   yanchor="bottom")),
    6: ("top-right", "lower right",  (1.0,   1.02), dict(x=1.0,   y=1.02, xanchor="right",  yanchor="bottom")),
    7: ("top-right", "lower right",  (1.0,   1.02), dict(x=1.0,   y=1.02, xanchor="right",  yanchor="bottom")),
    8: ("top",       "lower center", (0.5,   1.04), dict(x=0.5,   y=1.02, xanchor="center", yanchor="bottom")),
}

# Palette base colors per code — each is a list of RGB tuples
# Variation is applied per color via jitter at render time
PALETTE_BASES = {
    0:  [(0,   0,   0  )] * 6,                                          # black
    1:  [(30,  30,  30 ), (60,  60,  60 ), (90,  90,  90 ),
         (40,  50,  60 ), (50,  40,  70 ), (40,  70,  50 )],            # dark
    2:  [(31,  119, 180), (255, 127, 14 ), (44,  160, 44 ),
         (214, 39,  40 ), (148, 103, 189), (23,  190, 207)],            # bright
    3:  [(255, 140, 0  ), (230, 100, 0  ), (200, 80,  0  ),
         (255, 165, 50 ), (210, 120, 20 ), (240, 150, 30 )],            # orange
    4:  [(0,   30,  100), (0,   50,  130), (0,   70,  160),
         (20,  60,  140), (10,  40,  120), (30,  80,  150)],            # dark blue
    5:  [(50,  50,  50 ), (100, 100, 100), (150, 150, 150),
         (180, 180, 180), (200, 200, 200), (80,  80,  80 )],            # grey shades
    6:  [(0,   100, 200), (30,  130, 220), (60,  160, 240),
         (0,   80,  180), (20,  110, 210), (50,  140, 230)],            # bright blue
    7:  [(180, 220, 255), (150, 200, 240), (200, 230, 255),
         (160, 210, 245), (170, 215, 250), (140, 195, 235)],            # light multicolor
    8:  [(0,   0,   0  ), (0,   80,  160), (180, 30,  30 ),
         (0,   40,  120), (140, 20,  20 ), (20,  60,  140)],            # black, blue, red
    9:  [(20,  20,  60 ), (60,  20,  80 ), (20,  60,  40 ),
         (40,  40,  80 ), (80,  20,  60 ), (20,  80,  60 )],            # dark multicolor
    10: [(180, 0,   0  ), (0,   150, 0  ), (0,   0,   200),
         (160, 0,   0  ), (0,   130, 0  ), (0,   0,   180)],            # RGB
    11: [(180, 30,  30 ), (30,  80,  180), (180, 30,  30 ),
         (30,  80,  180), (160, 20,  20 ), (20,  60,  160)],            # red, blue
    12: [(100, 100, 120), (120, 100, 110), (110, 120, 100),
         (90,  110, 120), (115, 105, 95 ), (105, 115, 110)],            # muted multicolor
    13: [(0,   80,  160), (180, 80,  0  ), (200, 30,  30 ),
         (0,   60,  140), (160, 60,  0  ), (180, 20,  20 )],            # blue, red, orange
    14: [(0,   60,  160), (30,  90,  180), (60,  120, 200),
         (10,  70,  170), (40,  100, 190), (20,  80,  175)],            # blue shades
    15: [(180, 30,  30 ), (220, 100, 30 ), (60,  160, 60 ),
         (160, 20,  20 ), (200, 80,  20 ), (40,  140, 40 )],            # red, orange, green
    16: [(255, 255, 255)] * 6,                                          # white (dark bg)
    17: [(30,  160, 80 ), (0,   100, 180), (60,  180, 100),
         (20,  140, 60 ), (0,   80,  160), (40,  160, 90 )],            # green, blue
    18: [(31,  119, 180), (255, 127, 14 ), (44,  160, 44 ),
         (214, 39,  40 ), (148, 103, 189), (23,  190, 207)],            # fallback bright
}

AXIS_TEXT_COLORS = {0: "black", 1: None, 2: "green", 3: None, 4: "lightgray"}
AXIS_COLOR_BASES = {
    0: (0,   0,   0),    # black
    1: (80,  80,  80),   # dark gray
    2: (200, 200, 200),  # light gray
    4: (255, 255, 255),  # white (dark background)
}
GRIDLINE_COLOR_BASES = {
    1: (140, 140, 140),   # grey
    2: (0,   0,   0),     # black
    3: (200, 200, 200),   # light grey
    4: (31,  119, 180),   # blue
    5: (140, 140, 140),   # grey (dashed)
    6: (31,  119, 180),   # blue for dark bg
    7: (255, 255, 255),   # white for color bg
    8: (31,  119, 180),   # blue for light bg
}
LABEL_COLORS     = {0: "black", 1: None, 2: "black", 3: "bright"}

LINE_STRUCTURES   = {0: "straight", 1: "smooth"}
LINE_PATTERNS_MPL = {0: "-", 1: ":", 2: "--"}
LINE_PATTERNS_PLY = {0: "solid", 1: "dot", 2: "dash"}

POINT_SHAPE_MODES = {0: "dots", 1: "squares", 2: "by_line", 3: "none"}
POINT_SAME_COLORS = {0: "different", 1: "same", 2: "shade", 3: "none"}

# BRIGHT_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#17becf"]
# DARK_COLORS   = ["#111111", "#3a3a3a", "#555555", "#23415a", "#4a2c5f", "#31533b"]
# BLACK_COLORS  = ["#000000"] * 6
# PALETTE_TYPES = {0: BLACK_COLORS, 1: DARK_COLORS, 2: BRIGHT_COLORS}

BRIGHT_COLORS = [f"#{r:02x}{g:02x}{b:02x}" for r, g, b in PALETTE_BASES[2]]

MPL_MARKERS = {"dots": "o", "squares": "s", "by_line": "o", "none": None}
ALT_SHAPES  = {"dots": "circle", "squares": "square", "by_line": "circle"}
PLY_MARKERS = ["circle", "square", "triangle-up", "diamond", "cross", "x"]

X_MAJOR_TICK_VALUES = {0: "auto", 1: 1, 2: 2, 3: 5, 4: 10, 5: 25, 6: 50}
X_MINOR_TICK_VALUES = {0: 0, 1: 1, 2: 4, 3: 9}
Y_MAJOR_TICK_VALUES = {0: "auto", 1: 1, 2: 2, 3: 5, 4: 10, 5: 25, 6: 50, 7: 100, 8: 200}
Y_MINOR_TICK_VALUES = {0: 0, 1: 1, 2: 4, 3: 9}

In [35]:
# ── Helper functions ──────────────────────────────────────────────────────────

def _hex_to_rgb(hex_color: str) -> tuple[int, int, int]:
    hex_color = hex_color.lstrip("#")
    return int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)


def _relative_luminance(hex_color: str) -> float:
    """WCAG relative luminance of a hex color, in [0, 1]."""
    r, g, b = (_hex_to_rgb(hex_color))
    def linearize(c):
        c /= 255
        return c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4
    return 0.2126 * linearize(r) + 0.7152 * linearize(g) + 0.0722 * linearize(b)


def _contrast_ratio(hex_a: str, hex_b: str) -> float:
    """WCAG contrast ratio between two hex colors."""
    la = _relative_luminance(hex_a)
    lb = _relative_luminance(hex_b)
    lighter, darker = max(la, lb), min(la, lb)
    return (lighter + 0.05) / (darker + 0.05)

def _sample_with_contrast(base: tuple[int, int, int], bg: str,
                           rng: np.random.Generator, v: int = 30) -> str:
    check_contrast = bg not in {"none", "transparent"}
    for _ in range(20):
        r = int(np.clip(base[0] + rng.integers(-v, v + 1), 0, 255))
        g = int(np.clip(base[1] + rng.integers(-v, v + 1), 0, 255))
        b = int(np.clip(base[2] + rng.integers(-v, v + 1), 0, 255))
        candidate = f"#{r:02x}{g:02x}{b:02x}"
        if not check_contrast or _contrast_ratio(candidate, bg) >= 3.0:
            return candidate
    return "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"


def get_background_color(style: dict, rng: np.random.Generator | None = None) -> str:
    code = int(style.get("background", 1))
    base = BACKGROUND_COLOR_BASES.get(code)
    if base is None:
        return "none"
    rng = rng or np.random.default_rng()
    v   = 8  # small jitter — backgrounds should be subtle
    r = int(np.clip(base[0] + rng.integers(-v, v + 1), 0, 255))
    g = int(np.clip(base[1] + rng.integers(-v, v + 1), 0, 255))
    b = int(np.clip(base[2] + rng.integers(-v, v + 1), 0, 255))
    return f"#{r:02x}{g:02x}{b:02x}"


def get_foreground_color(style: dict) -> str:
    return "white" if int(style.get("background", 1)) == 3 else "black"


def get_title_color(style: dict, rng: np.random.Generator | None = None) -> str:
    code = int(style.get("title_color", 0))
    base = TITLE_COLOR_BASES.get(code, (0, 0, 0))
    rng  = rng or np.random.default_rng()
    bg   = get_background_color(style)

    # Skip contrast check for transparent background
    check_contrast = bg not in {"none", "transparent"}

    v = 30
    for _ in range(20):
        r = int(np.clip(base[0] + rng.integers(-v, v + 1), 0, 255))
        g = int(np.clip(base[1] + rng.integers(-v, v + 1), 0, 255))
        b = int(np.clip(base[2] + rng.integers(-v, v + 1), 0, 255))
        candidate = f"#{r:02x}{g:02x}{b:02x}"
        if not check_contrast or _contrast_ratio(candidate, bg) >= 3.0:
            return candidate

    # Fallback: return pure black or white depending on background luminance
    return "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"


def get_axis_color(style: dict, rng: np.random.Generator | None = None) -> str | None:
    """Returns the axis line color, or None for code 3 (no axes)."""
    code = int(style.get("axis_color", 0))
    if code == 3:
        return None
    base = AXIS_COLOR_BASES.get(code, (0, 0, 0))
    rng  = rng or np.random.default_rng()
    bg   = get_background_color(style)
    return _sample_with_contrast(base, bg, rng, v=20)

def get_legend_title_fontsize(style: dict, rng: np.random.Generator | None = None) -> int | None:
    if int(style.get("legend_title_size", 0)) != 1:
        return None
    rng = rng or np.random.default_rng()
    return int(np.clip(11 + rng.integers(-1, 2), 8, 16))


def get_title_fontsize(style: dict, rng: np.random.Generator | None = None) -> int:
    code = int(style.get("title_size", 0))
    base = TITLE_SIZE_BASES.get(code, 16)
    rng  = rng or np.random.default_rng()
    return int(np.clip(base + rng.integers(-2, 3), 8, 32))


def get_title_location(style: dict):
    """Returns (label, x_pos, halign, anchor)."""
    return TITLE_LOCATIONS.get(int(style.get("title_location", 1)),
                               ("center", 0.50, "center", "middle"))


def get_axis_text_color(style: dict, rng: np.random.Generator | None = None) -> str:
    code = int(style.get("axis_text_color", 0))
    if code == 1:
        return get_title_color(style, rng=rng)
    if code == 3:
        bg = get_background_color(style, rng=rng)
        return bg if bg not in {"none", "transparent"} else "white"
    return AXIS_TEXT_COLORS.get(code, "black") or "black"


def get_gridline_style(style: dict, rng: np.random.Generator | None = None) -> tuple[str, str]:
    """Returns (color, linestyle) where linestyle is 'solid' or 'dashed'."""
    code = int(style.get("gridline_color", 1))
    rng  = rng or np.random.default_rng()
    bg   = get_background_color(style)

    base = GRIDLINE_COLOR_BASES.get(code, (140, 140, 140))
    linestyle = "dashed" if code == 5 else "solid"

    color = _sample_with_contrast(base, bg, rng, v=15)
    return color, linestyle


def get_label_color(style: dict, rng: np.random.Generator | None = None,
                    line_color: str | None = None) -> str:
    code = int(style.get("label_color", 0))
    rng  = rng or np.random.default_rng()
    bg   = get_background_color(style)

    if code == 1:
        return get_title_color(style, rng=rng)

    if code == 2:
        return _sample_with_contrast((0, 0, 0), bg, rng, v=20)

    if code == 3:
        return line_color or BRIGHT_COLORS[0]

    if code == 4:
        base = 140
        v    = 40
        gray = int(np.clip(base + rng.integers(-v, v + 1), 80, 220))
        hex_gray = f"#{gray:02x}{gray:02x}{gray:02x}"
        # Retry if contrast is poor
        bg_check = bg not in {"none", "transparent"}
        for _ in range(20):
            gray = int(np.clip(base + rng.integers(-v, v + 1), 80, 220))
            hex_gray = f"#{gray:02x}{gray:02x}{gray:02x}"
            if not bg_check or _contrast_ratio(hex_gray, bg) >= 3.0:
                return hex_gray
        return "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"

    # code 0 or fallback — black with contrast check
    return _sample_with_contrast((0, 0, 0), bg, rng, v=10)

def get_legend_fill_and_outline(style: dict, rng: np.random.Generator | None = None) -> tuple[str | None, bool]:
    """Returns (fill_color, has_outline). Encodes all legend_fill codes in one place."""
    rng  = rng or np.random.default_rng()
    code = int(style.get("legend_fill", 0))

    if code == 0:
        return None, False

    if code == 1:
        # Gray fill, no outline — sample a gray shade for variation
        base = 180
        v    = 30
        gray = int(np.clip(base + rng.integers(-v, v + 1), 120, 235))
        return f"#{gray:02x}{gray:02x}{gray:02x}", False

    if code == 2:
        # No fill, no outline
        return None, False

    if code in {3, 4}:
        # White fill with black outline
        return "#ffffff", True

    return None, False


def get_legend_text_color(style: dict, rng: np.random.Generator | None = None) -> str | None:
    code = int(style.get("legend_text_color", 0))
    if code == 1:
        return get_title_color(style, rng=rng)
    if code == 2:
        return None

    base_color = {0: "#000000", 3: "#555555", 4: "#ffffff", 5: "#888888", 6: "#333333"}.get(code, "#000000")

    bg = get_background_color(style)
    if bg in {"none", "transparent"}:
        return base_color

    r, g, b = _hex_to_rgb(base_color)
    rng = rng or np.random.default_rng()
    v = 20
    for _ in range(20):
        rc = int(np.clip(r + rng.integers(-v, v + 1), 0, 255))
        gc = int(np.clip(g + rng.integers(-v, v + 1), 0, 255))
        bc = int(np.clip(b + rng.integers(-v, v + 1), 0, 255))
        candidate = f"#{rc:02x}{gc:02x}{bc:02x}"
        if _contrast_ratio(candidate, bg) >= 3.0:
            return candidate

    return "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"

LEGEND_TITLE_COLOR_BASES = {
    0: (0, 0, 0),        # black
    1: (100, 100, 100),  # gray
    2: (255, 255, 255),  # white
}

def get_legend_title_color(style: dict, rng: np.random.Generator | None = None) -> str:
    code = int(style.get("legend_title_color", 0))
    base = LEGEND_TITLE_COLOR_BASES.get(code, (0, 0, 0))
    rng  = rng or np.random.default_rng()
    bg   = get_background_color(style)

    check_contrast = bg not in {"none", "transparent"}

    v = 20
    for _ in range(20):
        r = int(np.clip(base[0] + rng.integers(-v, v + 1), 0, 255))
        g = int(np.clip(base[1] + rng.integers(-v, v + 1), 0, 255))
        b = int(np.clip(base[2] + rng.integers(-v, v + 1), 0, 255))
        candidate = f"#{r:02x}{g:02x}{b:02x}"
        if not check_contrast or _contrast_ratio(candidate, bg) >= 3.0:
            return candidate

    return "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"


def color_list(style: dict, n: int, rng: np.random.Generator | None = None) -> list:
    code  = int(style.get("palette_type", 2))
    rng   = rng or np.random.default_rng()
    bases = PALETTE_BASES.get(code, PALETTE_BASES[2])
    bg    = get_background_color(style)
    v     = 15

    def _too_similar(candidate: str, existing: list[str], threshold: int = 60) -> bool:
        cr, cg, cb = _hex_to_rgb(candidate)
        for ex in existing:
            er, eg, eb = _hex_to_rgb(ex)
            if abs(cr - er) + abs(cg - eg) + abs(cb - eb) < threshold:
                return True
        return False

    colors = []
    for i in range(n):
        base = bases[i % len(bases)]
        chosen = None
        for _ in range(40):
            r = int(np.clip(base[0] + rng.integers(-v, v + 1), 0, 255))
            g = int(np.clip(base[1] + rng.integers(-v, v + 1), 0, 255))
            b = int(np.clip(base[2] + rng.integers(-v, v + 1), 0, 255))
            candidate = f"#{r:02x}{g:02x}{b:02x}"
            bg_ok      = bg in {"none", "transparent"} or _contrast_ratio(candidate, bg) >= 2.0
            similar_ok = not _too_similar(candidate, colors)
            if bg_ok and similar_ok:
                chosen = candidate
                break
        if chosen is None:
            # Fallback — spread evenly through hue space
            chosen = "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"
        colors.append(chosen)

    return colors


def line_pattern_for(style: dict, idx: int, renderer: str = "mpl") -> str:
    code = int(style.get("line_pattern", 0))
    if code == 3:   # mixed
        sub = idx % 3
        return LINE_PATTERNS_MPL.get(sub, "-") if renderer == "mpl" else LINE_PATTERNS_PLY.get(sub, "solid")
    return LINE_PATTERNS_MPL.get(code, "-") if renderer == "mpl" else LINE_PATTERNS_PLY.get(code, "solid")


def marker_for(style: dict, idx: int):
    mode = POINT_SHAPE_MODES.get(int(style.get("point_shape_mode", 0)), "dots")
    if mode == "none":
        return None
    if mode == "by_line":
        return ["o", "s", "^", "D", "P", "X"][idx % 6]
    return MPL_MARKERS.get(mode, "o")


def point_color(line_color: str, style: dict, idx: int) -> str:
    mode = POINT_SAME_COLORS.get(int(style.get("point_same_color", 1)), "same")
    if mode == "none":
        return line_color
    if mode == "different":
        return BRIGHT_COLORS[(idx + 2) % len(BRIGHT_COLORS)]
    if mode == "shade":
        return "#999999"
    return line_color


def legend_location_mpl(style: dict):
    entry = LEGEND_ORIENTATIONS.get(int(style.get("legend_orientation", 2)))
    if not entry:
        return None, None
    _, loc, bbox, _ = entry
    return loc, bbox

def legend_orient_altair(style: dict):
    entry = LEGEND_ORIENTATIONS.get(int(style.get("legend_orientation", 2)))
    return entry[0] if entry else None

def legend_coords_plotly(style: dict) -> dict:
    entry = LEGEND_ORIENTATIONS.get(int(style.get("legend_orientation", 2)))
    return entry[3] if entry else {}

def legend_orient_altair(style: dict):
    entry = LEGEND_ORIENTATIONS.get(int(style.get("legend_orientation", 2)))
    return entry[0] if entry else None


def apply_gridlines_mpl(ax, style: dict, rng: np.random.Generator | None = None):
    grid = int(style.get("gridlines", 0))
    if grid == 0:
        ax.grid(False)
        return

    color, linestyle = get_gridline_style(style, rng=rng)
    mpl_linestyle = "--" if linestyle == "dashed" else "-"

    if grid == 2:
        ax.yaxis.grid(True, color=color, linewidth=0.6, linestyle=mpl_linestyle, alpha=0.75, zorder=0)
        ax.xaxis.grid(False)
    elif grid in {1, 3, 4}:
        width = 0.4 if grid == 3 else 0.8 if grid == 4 else 0.6
        ax.grid(True, color=color, linewidth=width, linestyle=mpl_linestyle, alpha=0.75, zorder=0)
    else:
        ax.grid(False)


def apply_outline_mpl(ax, fig, style: dict, rng: np.random.Generator | None = None):
    outline  = int(style.get("chart_outline", 1))
    fg       = get_foreground_color(style)
    axis_c   = get_axis_color(style, rng=rng)

    if axis_c is None:
        # code 3 — no axes at all
        for s in ax.spines.values():
            s.set_visible(False)
        ax.tick_params(left=False, bottom=False)
        if int(style.get("image_outline", 0)) == 1:
            fig.patch.set_edgecolor(fg)
            fig.patch.set_linewidth(1.2)
        return

    for spine in ax.spines.values():
        spine.set_color(axis_c)

    if outline == 0:
        for s in ax.spines.values():
            s.set_visible(False)
    elif outline == 1:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    elif outline == 2:
        pass  # full box
    elif outline == 4:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_visible(False)
        ax.yaxis.set_visible(False)

    if int(style.get("image_outline", 0)) == 1:
        fig.patch.set_edgecolor(fg)
        fig.patch.set_linewidth(1.2)

In [36]:
# ── Tick helpers ──────────────────────────────────────────────────────────────

def _resolve_major_interval(code, axis: str):
    """Convert a style major-tick code to a concrete numeric interval, or None for auto."""
    table = X_MAJOR_TICK_VALUES if axis == "x" else Y_MAJOR_TICK_VALUES
    val   = table.get(int(code), "auto")
    if val == "auto":
        return None
    try:
        v = float(val)
        return v if v > 0 else None
    except (TypeError, ValueError):
        return None

def apply_tick_settings_mpl(ax, style: dict, x_positions=None):
    from matplotlib.ticker import AutoLocator, NullLocator, AutoMinorLocator

    show_x_major = int(style.get("x_major_ticks", 1)) == 1
    show_y_major = int(style.get("y_major_ticks", 1)) == 1
    show_x_minor = int(style.get("x_minor_ticks", 0)) == 1
    show_y_minor = int(style.get("y_minor_ticks", 0)) == 1

    if show_x_major:
        if x_positions is not None and len(x_positions) > 0:
            pass  # x ticks already set via set_xticks
        else:
            ax.xaxis.set_major_locator(AutoLocator())
        if show_x_minor:
            ax.xaxis.set_minor_locator(AutoMinorLocator())
        else:
            ax.xaxis.set_minor_locator(NullLocator())
    else:
        ax.xaxis.set_major_locator(NullLocator())
        ax.xaxis.set_minor_locator(NullLocator())
        ax.set_xticklabels([])

    if show_y_major:
        ax.yaxis.set_major_locator(AutoLocator())
        if show_y_minor:
            ax.yaxis.set_minor_locator(AutoMinorLocator())
        else:
            ax.yaxis.set_minor_locator(NullLocator())
    else:
        ax.yaxis.set_major_locator(NullLocator())
        ax.yaxis.set_minor_locator(NullLocator())
        ax.set_yticklabels([])


def apply_tick_settings_plotly(fig, style: dict):
    show_x_major = int(style.get("x_major_ticks", 1)) == 1
    show_y_major = int(style.get("y_major_ticks", 1)) == 1
    show_x_minor = int(style.get("x_minor_ticks", 0)) == 1
    show_y_minor = int(style.get("y_minor_ticks", 0)) == 1

    fig.update_xaxes(
        showticklabels=show_x_major,
        ticks="outside" if show_x_major else "",
        minor=dict(ticks="outside", showgrid=False) if show_x_minor else {},
    )
    fig.update_yaxes(
        showticklabels=show_y_major,
        ticks="outside" if show_y_major else "",
        minor=dict(ticks="outside", showgrid=False) if show_y_minor else {},
    )


def apply_tick_settings_altair(x_enc, y_enc, style: dict):
    show_x_major = int(style.get("x_major_ticks", 1)) == 1
    show_y_major = int(style.get("y_major_ticks", 1)) == 1
    show_x_minor = int(style.get("x_minor_ticks", 0)) == 1
    show_y_minor = int(style.get("y_minor_ticks", 0)) == 1

    if not show_x_major:
        x_enc = x_enc.axis(alt.Axis(ticks=False, labels=False))
    elif show_x_minor:
        x_enc = x_enc.axis(alt.Axis(tickCount=20))

    if not show_y_major:
        y_enc = y_enc.axis(alt.Axis(ticks=False, labels=False))
    elif show_y_minor:
        y_enc = y_enc.axis(alt.Axis(tickCount=20))

    return x_enc, y_enc

## 4. Title generation

In [ ]:

# UNCOMMENT FOR GROQ
#_groq_client = Groq() #COMMENT OUT

MAX_TITLE_CHARS = 60


def make_title(context: dict, style: dict | None = None) -> tuple[str, str | None]:
    x_label = context.get("x_label", "X")
    y_label = context.get("y_label", "Value")
    n_lines = context.get("n_lines", 1)

    need_subtitle = style is not None and int(style.get("subtitle_present", 0)) == 1

    system_prompt = (
        "You generate short, realistic chart titles for line charts — the kind you'd see "
        "in a business report, scientific paper, or dashboard. "
        "Keep titles under 60 characters. Keep subtitles under 60 characters. "
        "Be concise and natural. No quotes, no markdown."
    )
    user_prompt = (
        f"Generate a chart title for a line chart showing {y_label} over {x_label}"
        + (f" across {n_lines} series" if n_lines > 1 else "")
        + ".\n"
    )
    if need_subtitle:
        user_prompt += (
            "Also generate a short subtitle (one line, adds context or detail). "
            "Respond in this exact format:\n"
            "TITLE: <title here>\n"
            "SUBTITLE: <subtitle here>"
        )
    else:
        user_prompt += "Respond with just the title, nothing else."

    try:
        response = ollama.chat(
            model="llama3.2",  # or whichever model you have pulled
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
        )
        raw = response["message"]["content"].strip()

        if need_subtitle and "TITLE:" in raw:
            lines = {k.strip(): v.strip() for k, v in
                     (line.split(":", 1) for line in raw.splitlines() if ":" in line)}
            return lines.get("TITLE", raw)[:MAX_TITLE_CHARS], lines.get("SUBTITLE", "")[:MAX_TITLE_CHARS]

        return raw[:MAX_TITLE_CHARS], None

    except Exception:
        fallback     = f"{y_label} over {x_label}"[:MAX_TITLE_CHARS]
        fallback_sub = "Synthetic data" if need_subtitle else None
        return fallback, fallback_sub

## Harmonize styles

In [38]:
def harmonize_style(style: dict) -> dict:
    style = dict(style)

    bg = int(style.get("background", 1))
    dark_backgrounds = {3}   # dark_gray code

    # Title consistency
    if int(style.get("title_present", 1)) == 0:
        style["title_location"]   = 0
        style["subtitle_present"] = 0
        style["title_size"]       = 0
        style["title_color"]      = 0
    if int(style.get("title_location", 1)) == 0:
        style["title_present"]    = 0
        style["subtitle_present"] = 0

    # Legend consistency
    if int(style.get("legend_present", 1)) == 0 or int(style.get("legend_orientation", 2)) == 0:
        style["legend_present"]     = 0
        style["legend_orientation"] = 0
        style["legend_title_size"]  = 0
        style["legend_title_color"] = 0
        style["legend_text_color"]  = 2
        style["legend_outline"]     = 0
        style["legend_fill"]        = 0

    # Label consistency
    if int(style.get("direct_labels", 0)) == 0 or int(style.get("label_content", 0)) == 0:
        style["direct_labels"] = 0
        style["label_content"] = 0

    # Gridline consistency
    if int(style.get("gridlines", 0)) == 0:
        style["gridline_color"] = 0

    # Point consistency
    mode = int(style.get("point_shape_mode", 0))
    if mode == 3:   # none
        style["point_same_color"] = 3
    if int(style.get("point_same_color", 1)) == 3:
        style["point_shape_mode"] = 3

    # Coerce ints
    style["n_lines"] = int(style.get("n_lines", 1))

    # # Title color must be readable against background
    # title_color = int(style.get("title_color", 0))
    # if bg in dark_backgrounds and title_color != 6:
    #     style["title_color"] = 6   # force white
    # elif bg not in dark_backgrounds and title_color == 6:
    #     style["title_color"] = 0   # force black

    # # Axis text color must be readable against background
    # axis_text = int(style.get("axis_text_color", 0))
    # if bg in dark_backgrounds and axis_text == 0:
    #     style["axis_text_color"] = 4   # force white on dark background
    # elif bg not in dark_backgrounds and axis_text == 4:
    #     style["axis_text_color"] = 0   # force black on light background

    # # Legend text color must be readable against background
    # if int(style.get("legend_present", 1)) == 1:
    #     legend_text = int(style.get("legend_text_color", 0))
    #     if bg in dark_backgrounds and legend_text in {0, 3}:
    #         style["legend_text_color"] = 4   # force white
    #     elif bg not in dark_backgrounds and legend_text == 4:
    #         style["legend_text_color"] = 0   # force black

    # No manual dark_backgrounds override needed — get_title_color handles
    # contrast via luminance checking at render time. Nothing to do here.

    return style

## 5. Line renderers

In [39]:
def render_line_altair(df: pd.DataFrame, context: dict, title: str,
                       subtitle: str | None, style: dict,
                       rng: np.random.Generator | None = None) -> alt.Chart:
    rng             = rng or np.random.default_rng()
    fg              = get_foreground_color(style)
    bg              = get_background_color(style, rng=rng)
    axis_color      = get_axis_text_color(style, rng=rng)
    axis_line_color = get_axis_color(style, rng=rng)
    orient          = int(style.get("axis_text_orientation", 0))

    series_names = list(df["series"].unique())
    colors       = color_list(style, len(series_names), rng=rng)

    show_x_labels = int(style.get("x_axis_month_day_labels", 0)) != 0

    x_title = context["x_label"] if orient in {0, 1, 3, 4} else None
    y_title = context["y_label"] if orient in {0, 1, 3, 5} else None

    if show_x_labels:
        x_enc       = alt.X("x_label:N", sort=context.get("x_tick_labels"), title=x_title)
        label_x_enc = alt.X("x_label:N", sort=context.get("x_tick_labels"), title=None)
    else:
        x_enc       = alt.X("x:Q", title=x_title)
        label_x_enc = alt.X("x:Q", title=None)

    y_enc = alt.Y("y:Q", title=None, scale=alt.Scale(zero=False))

    if not show_x_labels:
        x_enc, y_enc = apply_tick_settings_altair(x_enc, y_enc, style)

    legend_orient = legend_orient_altair(style)
    show_legend   = int(style.get("legend_present", 1)) == 1 and legend_orient is not None

    legend_obj = None
    if show_legend and legend_orient:
        legend_title_fs = get_legend_title_fontsize(style, rng=rng)
        legend_obj = alt.Legend(
            orient=legend_orient,
            title="Series" if legend_title_fs is not None else None,
            titleFontSize=legend_title_fs if legend_title_fs is not None else alt.Undefined,
            labelColor=get_legend_text_color(style) or fg,
            titleColor=get_legend_title_color(style, rng=rng),
        )

    grid      = int(style.get("gridlines", 0))
    x_grid    = grid in {1, 3, 4}
    y_grid    = grid in {1, 2, 3, 4}

    gridcolor, grid_linestyle = get_gridline_style(style, rng=rng)
    grid_dash = [4, 4] if grid_linestyle == "dashed" else []

    outline       = int(style.get("chart_outline", 1))
    view_stroke   = fg if outline == 2 else "transparent"

    # axis_line_color None means code 3 (no axes)
    show_x_domain = outline != 0 and axis_line_color is not None
    show_y_domain = outline in {1, 2} and axis_line_color is not None
    domain_color  = axis_line_color or "transparent"

    x_axis = alt.Axis(
        labelColor=axis_color, titleColor=axis_color,
        grid=x_grid, gridColor=gridcolor, gridOpacity=0.8, gridDash=grid_dash, zindex=0,
        domain=show_x_domain, ticks=show_x_domain,
        labelAngle=0,
        labelOverlap=True,
    )
    y_axis = alt.Axis(
        labelColor=axis_color, titleColor=axis_color,
        grid=y_grid, gridColor=gridcolor, gridOpacity=0.8, gridDash=grid_dash, zindex=0,
        domain=show_y_domain, ticks=show_y_domain,
        labels=show_y_domain,
        title=y_title if show_y_domain else None,
    )

    x_enc = x_enc.axis(x_axis)
    y_enc = y_enc.axis(y_axis)

    color_scale = alt.Scale(domain=series_names, range=colors)
    interp      = "monotone" if LINE_STRUCTURES.get(int(style.get("line_structure", 0))) == "smooth" else "linear"

    if int(style.get("line_pattern", 0)) == 3:
        line = alt.Chart(df).mark_line(interpolate=interp).encode(
            x=x_enc, y=y_enc,
            color=alt.Color("series:N", scale=color_scale, legend=legend_obj),
            strokeDash=alt.StrokeDash("series:N"),
        )
    else:
        line = alt.Chart(df).mark_line(interpolate=interp).encode(
            x=x_enc, y=y_enc,
            color=alt.Color("series:N", scale=color_scale, legend=legend_obj),
        )

    layers = [line]

    if int(style.get("point_shape_mode", 0)) != 3:
        mode      = POINT_SHAPE_MODES.get(int(style.get("point_shape_mode", 0)), "dots")
        shape_enc = alt.Shape("series:N") if mode == "by_line" else alt.value(
            "square" if mode == "squares" else "circle"
        )
        layers.append(
            alt.Chart(df).mark_point(size=65, filled=True).encode(
                x=x_enc, y=y_enc,
                color=alt.Color("series:N", scale=color_scale, legend=None),
                shape=shape_enc,
            )
        )

    if int(style.get("direct_labels", 0)) != 0 and int(style.get("label_content", 0)) != 0:
        if int(style.get("direct_labels", 0)) == 1:
            label_df = df.copy()
        else:
            parts = []
            for sname in df["series"].unique():
                sdf = df[df["series"] == sname].copy()
                parts.append(sdf.iloc[::max(1, len(sdf) // 4)].copy())
            label_df = pd.concat(parts, ignore_index=True) if parts else df.iloc[0:0].copy()

        lc_code           = int(style.get("label_content", 0))
        label_df["label"] = (label_df["series"].astype(str) if lc_code == 1
                             else label_df["y"].round(1).astype(str))

        layers.append(
            alt.Chart(label_df).mark_text(
                align="left", dx=4, dy=-4, fontSize=9,
                color=get_label_color(style, rng=rng),
            ).encode(x=label_x_enc, y=alt.Y("y:Q", title=None), text=alt.Text("label:N"))
        )

    title_params = alt.TitleParams(text="", subtitle="")
    if int(style.get("title_present", 1)) == 1 and title:
        sub = subtitle if (int(style.get("subtitle_present", 0)) == 1 and subtitle) else ""
        _, _, _, anchor = get_title_location(style)
        title_params = alt.TitleParams(
            text=title, subtitle=sub, anchor=anchor,
            fontSize=get_title_fontsize(style, rng=rng),
            color=get_title_color(style, rng=rng),
            subtitleColor=fg,
        )

    chart = (
        alt.layer(*layers)
        .properties(width=640, height=390, title=title_params)
        .configure(
            background=bg if bg != "none" else alt.Undefined,
            axis=alt.AxisConfig(
                domainColor=domain_color, tickColor=domain_color,
                labelColor=axis_color, titleColor=axis_color,
            ),
            legend=alt.LegendConfig(
                strokeColor=fg if int(style.get("legend_outline", 0)) == 1 else "transparent",
                padding=6,
                fillColor=get_legend_fill_and_outline(style, rng=rng)[0] or "transparent",
            ),
            view=alt.ViewConfig(stroke=view_stroke),
        )
    )
    return chart

In [40]:
def render_line_matplotlib(df: pd.DataFrame, context: dict, title: str,
                           subtitle: str | None, style: dict,
                           rng: np.random.Generator | None = None):
    rng         = rng or np.random.default_rng()
    bg          = get_background_color(style, rng=rng)
    fg          = get_foreground_color(style)
    orient_code = int(style.get("legend_orientation", 2))

    fig, ax = plt.subplots(figsize=(7.2, 4.6), dpi=140)
    fig.patch.set_facecolor((0, 0, 0, 0) if bg == "none" else bg)
    ax.set_facecolor((0, 0, 0, 0) if bg == "none" else bg)
    ax.set_axisbelow(True)

    series_names = list(df["series"].unique())
    colors       = color_list(style, len(series_names), rng=rng)

    for idx, series in enumerate(series_names):
        sub    = df[df["series"] == series].sort_values("x")
        lc     = colors[idx]
        marker = marker_for(style, idx)
        pc     = point_color(lc, style, idx)
        pat    = line_pattern_for(style, idx, "mpl")

        if LINE_STRUCTURES.get(int(style.get("line_structure", 0))) == "smooth" and len(sub) >= 4:
            xs = np.linspace(sub["x"].min(), sub["x"].max(), 160)
            ys = np.interp(xs, sub["x"], sub["y"])
            ax.plot(xs, ys, linestyle=pat, color=lc, linewidth=2, label=series)
            if marker is not None:
                ax.scatter(sub["x"], sub["y"], marker=marker, color=pc, s=28, zorder=3)
        else:
            ax.plot(sub["x"], sub["y"], linestyle=pat, color=lc,
                    marker=marker, markerfacecolor=pc, markeredgecolor=pc,
                    linewidth=2, label=series)

        if int(style.get("direct_labels", 0)) != 0 and int(style.get("label_content", 0)) != 0:
            lc_code   = int(style.get("label_content", 0))
            max_shown = max(1, 8 // max(1, len(series_names)))
            if int(style.get("direct_labels", 0)) == 1:
                step   = max(1, len(sub) // max_shown)
                points = sub.iloc[::step]
            else:
                points = sub.iloc[::max(1, len(sub) // 4)]
            for _, row in points.iterrows():
                text    = series if lc_code == 1 else f"{row['y']:.1f}"
                label_c = get_label_color(style, rng=rng, line_color=lc)
                ax.text(row["x"], row["y"], text, fontsize=7, color=label_c, ha="left", va="bottom")

    if context.get("x_tick_positions") and context.get("x_tick_labels"):
        positions = list(context["x_tick_positions"])
        labels    = list(context["x_tick_labels"])
        if len(positions) > 1:
            avg_chars  = max(1, sum(len(str(l)) for l in labels) / len(labels))
            fig_width  = fig.get_figwidth()
            max_labels = max(2, int(fig_width * 72 / (avg_chars * 7.5)))
            if len(positions) > max_labels:
                step      = max(1, len(positions) // max_labels)
                positions = positions[::step]
                labels    = labels[::step]
        ax.set_xticks(positions)
        ax.set_xticklabels(labels, rotation=0, ha="center")

    axis_color = get_axis_text_color(style, rng=rng)
    orient     = int(style.get("axis_text_orientation", 0))
    ax.tick_params(axis="both", colors=axis_color)

    if orient in {0, 1, 3, 4, 6}:
        ax.set_xlabel(context["x_label"], color=axis_color, rotation=0)
    else:
        ax.set_xlabel("")

    if orient in {0, 3, 5}:
        ax.set_ylabel(context["y_label"], color=axis_color, rotation=90)
    elif orient == 1:
        ax.set_ylabel(context["y_label"], color=axis_color, rotation=0, labelpad=40)
    else:
        ax.set_ylabel("")

    apply_gridlines_mpl(ax, style, rng=rng)
    apply_outline_mpl(ax, fig, style, rng=rng)
    ax.relim()
    ax.autoscale_view()
    apply_tick_settings_mpl(ax, style, x_positions=context.get("x_tick_positions"))

    if int(style.get("title_present", 1)) == 1 and title:
        _, _, halign, _ = get_title_location(style)
        title_text = title if not (int(style.get("subtitle_present", 0)) == 1 and subtitle) \
                           else f"{title}\n{subtitle}"
        ax.set_title(title_text, loc=halign,
                     color=get_title_color(style, rng=rng),
                     fontsize=get_title_fontsize(style, rng=rng), pad=10)

    if int(style.get("legend_present", 1)) == 1 and orient_code != 0:
        entry = LEGEND_ORIENTATIONS.get(orient_code)
        if entry:
            _, loc, bbox, _ = entry
            fill_color, has_outline = get_legend_fill_and_outline(style, rng=rng)
            has_outline = has_outline or int(style.get("legend_outline", 0)) == 1
            ncol        = len(series_names) if orient_code in {3, 4, 8} else 1
            leg_kwargs  = {"frameon": has_outline or fill_color is not None}
            if fill_color:
                leg_kwargs["facecolor"] = fill_color
            legend_title_fs = get_legend_title_fontsize(style, rng=rng)
            leg = ax.legend(
                title="Series" if legend_title_fs is not None else None,
                loc=loc, bbox_to_anchor=bbox, ncol=ncol, **leg_kwargs,
            )
            if leg and legend_title_fs is not None:
                leg.get_title().set_fontsize(legend_title_fs)
                leg.get_title().set_color(get_legend_title_color(style, rng=rng))
            if leg:
                frame = leg.get_frame()
                if has_outline:
                    frame.set_edgecolor(fg); frame.set_linewidth(1.2)
                else:
                    frame.set_edgecolor("none")
                txt_color = get_legend_text_color(style, rng=rng)
                if txt_color:
                    for txt in leg.get_texts():
                        txt.set_color(txt_color)

    if orient_code in {3, 8}:
        fig.tight_layout(); plt.subplots_adjust(top=0.82)
    elif orient_code == 4:
        fig.tight_layout(); plt.subplots_adjust(bottom=0.25)
    elif orient_code in {5, 6, 7}:
        fig.tight_layout(); plt.subplots_adjust(top=0.82)
    elif orient_code == 1:
        fig.tight_layout(); plt.subplots_adjust(left=0.28)
    elif orient_code == 2:
        fig.tight_layout(); plt.subplots_adjust(right=0.72)
    else:
        fig.tight_layout()
    return fig


def render_line_seaborn(df: pd.DataFrame, context: dict, title: str,
                        subtitle: str | None, style: dict,
                        rng: np.random.Generator | None = None):
    """Seaborn renderer — delegates most styling to the Matplotlib helpers."""
    if sns is None:
        raise ImportError("seaborn is not installed")
    rng             = rng or np.random.default_rng()
    bg              = get_background_color(style, rng=rng)
    fg              = get_foreground_color(style)
    orient_code     = int(style.get("legend_orientation", 2))
    legend_title_fs = get_legend_title_fontsize(style, rng=rng)

    series_names = list(df["series"].unique())
    colors       = color_list(style, len(series_names), rng=rng)
    palette      = {s: colors[i] for i, s in enumerate(series_names)}

    fig, ax = plt.subplots(figsize=(7.2, 4.6), dpi=140)
    fig.patch.set_facecolor((0, 0, 0, 0) if bg == "none" else bg)
    ax.set_facecolor((0, 0, 0, 0) if bg == "none" else bg)
    ax.set_axisbelow(True)

    use_markers = int(style.get("point_shape_mode", 0)) != 3
    use_dashes  = int(style.get("line_pattern", 0)) != 0

    show_legend = int(style.get("legend_present", 1)) == 1
    sns.lineplot(
        data=df.sort_values(["series", "x"]),
        x="x", y="y", hue="series",
        style="series" if (use_markers or use_dashes) else None,
        markers=use_markers, dashes=use_dashes,
        palette=palette, linewidth=2, ax=ax,
        legend=show_legend,
    )

    if show_legend:
        loc_mpl, bbox_mpl = legend_location_mpl(style)
        if loc_mpl is not None:
            ncol = len(series_names) if orient_code in {3, 4, 8} else 1
            fill_color, fill_outline = get_legend_fill_and_outline(style, rng=rng)
            has_outline = fill_outline or int(style.get("legend_outline", 0)) == 1
            leg_kwargs = {"frameon": has_outline or fill_color is not None}
            if fill_color:
                leg_kwargs["facecolor"] = fill_color
            ax.legend(loc=loc_mpl, bbox_to_anchor=bbox_mpl, ncol=ncol, **leg_kwargs)
        leg = ax.get_legend()
        if leg:
            fill_color, fill_outline = get_legend_fill_and_outline(style, rng=rng)
            has_outline = fill_outline or int(style.get("legend_outline", 0)) == 1
            frame = leg.get_frame()
            if fill_color:
                frame.set_facecolor(fill_color)
            if has_outline:
                frame.set_edgecolor(fg)
                frame.set_linewidth(1.2)
            else:
                frame.set_edgecolor("none")
            if legend_title_fs is not None:
                leg.set_title("Series")
                leg.get_title().set_fontsize(legend_title_fs)
                leg.get_title().set_color(get_legend_title_color(style, rng=rng))
            else:
                leg.set_title(None)
            txt_color = get_legend_text_color(style, rng=rng)
            if txt_color:
                for txt in leg.get_texts():
                    txt.set_color(txt_color)

    if int(style.get("direct_labels", 0)) != 0 and int(style.get("label_content", 0)) != 0:
        lc_code   = int(style.get("label_content", 0))
        max_shown = max(1, 8 // max(1, len(series_names)))
        for series in series_names:
            sub = df[df["series"] == series].sort_values("x")
            lc  = colors[series_names.index(series)]
            if int(style.get("direct_labels", 0)) == 1:
                step   = max(1, len(sub) // max_shown)
                points = sub.iloc[::step]
            else:
                points = sub.iloc[::max(1, len(sub) // 4)]
            for _, row in points.iterrows():
                text = series if lc_code == 1 else f"{row['y']:.1f}"
                ax.text(row["x"], row["y"], text, fontsize=8,
                        color=get_label_color(style, rng=rng, line_color=lc))

    if context.get("x_tick_positions") and context.get("x_tick_labels"):
        positions = list(context["x_tick_positions"])
        labels    = list(context["x_tick_labels"])
        if len(positions) > 1:
            avg_chars  = max(1, sum(len(str(l)) for l in labels) / len(labels))
            fig_width  = fig.get_figwidth()
            max_labels = max(2, int(fig_width * 72 / (avg_chars * 7.5)))
            if len(positions) > max_labels:
                step      = max(1, len(positions) // max_labels)
                positions = positions[::step]
                labels    = labels[::step]
        ax.set_xticks(positions)
        ax.set_xticklabels(labels, rotation=0, ha="center")

    axis_color = get_axis_text_color(style, rng=rng)
    orient     = int(style.get("axis_text_orientation", 0))
    ax.tick_params(axis="both", colors=axis_color)

    if orient in {0, 1, 3, 4, 6}:
        ax.set_xlabel(context["x_label"], color=axis_color, rotation=0)
    else:
        ax.set_xlabel("")

    if orient in {0, 3, 5}:
        ax.set_ylabel(context["y_label"], color=axis_color, rotation=90)
    elif orient == 1:
        ax.set_ylabel(context["y_label"], color=axis_color, rotation=0, labelpad=40)
    else:
        ax.set_ylabel("")

    apply_gridlines_mpl(ax, style, rng=rng)
    apply_outline_mpl(ax, fig, style, rng=rng)
    apply_tick_settings_mpl(ax, style, x_positions=context.get("x_tick_positions"))

    if int(style.get("title_present", 1)) == 1 and title:
        _, _, halign, _ = get_title_location(style)
        title_text = title if not (int(style.get("subtitle_present", 0)) == 1 and subtitle) \
                           else f"{title}\n{subtitle}"
        ax.set_title(title_text, loc=halign,
                     color=get_title_color(style, rng=rng),
                     fontsize=get_title_fontsize(style, rng=rng), pad=10)

    if orient_code in {3, 8}:
        fig.tight_layout(); plt.subplots_adjust(top=0.82)
    elif orient_code == 4:
        fig.tight_layout(); plt.subplots_adjust(bottom=0.25)
    elif orient_code in {5, 6, 7}:
        fig.tight_layout(); plt.subplots_adjust(top=0.82)
    elif orient_code == 1:
        fig.tight_layout(); plt.subplots_adjust(left=0.28)
    elif orient_code == 2:
        fig.tight_layout(); plt.subplots_adjust(right=0.72)
    else:
        fig.tight_layout()
    return fig

In [41]:
def render_line_plotly(df: pd.DataFrame, context: dict, title: str,
                       subtitle: str | None, style: dict,
                       rng: np.random.Generator | None = None):
    rng          = rng or np.random.default_rng()
    series_names = list(df["series"].unique())
    colors       = color_list(style, len(series_names), rng=rng)

    fig = go.Figure()
    for idx, series in enumerate(series_names):
        sub  = df[df["series"] == series].sort_values("x")
        lc   = colors[idx]
        pat  = line_pattern_for(style, idx, "plotly")
        mkr  = marker_for(style, idx)
        mode = "lines" if mkr is None else "lines+markers"

        text = None
        if int(style.get("direct_labels", 0)) != 0 and int(style.get("label_content", 0)) != 0:
            lc_code   = int(style.get("label_content", 0))
            all_text  = [series if lc_code == 1 else f"{v:.1f}" for v in sub["y"]]
            n_points  = len(all_text)
            max_shown = max(1, min(n_points, 8 // max(1, len(series_names))))
            step      = max(1, n_points // max_shown)
            text      = [t if i % step == 0 else "" for i, t in enumerate(all_text)]
            mode     += "+text"

        shape = "spline" if LINE_STRUCTURES.get(int(style.get("line_structure", 0))) == "smooth" else "linear"

        fig.add_trace(go.Scatter(
            x=sub["x"], y=sub["y"], name=series, mode=mode,
            line={"color": lc, "dash": pat, "shape": shape},
            marker={"symbol": PLY_MARKERS[idx % len(PLY_MARKERS)] if mkr else "circle",
                    "color": point_color(lc, style, idx) or lc},
            text=text, textposition="top center",
            textfont={"color": get_label_color(style, rng=rng, line_color=lc)},
        ))

    bg         = get_background_color(style, rng=rng)
    fg         = get_foreground_color(style)
    paper_bg   = "rgba(0,0,0,0)" if bg == "none" else bg
    axis_color = get_axis_text_color(style, rng=rng)
    orient     = int(style.get("axis_text_orientation", 0))

    title_text = None
    if int(style.get("title_present", 1)) == 1 and title:
        title_text = title if not (int(style.get("subtitle_present", 0)) == 1 and subtitle) \
                           else f"{title}<br><sup>{subtitle}</sup>"
    _, x_pos, _, _alt_anchor = get_title_location(style)
    x_anchor = {"start": "left", "end": "right", "middle": "center"}.get(_alt_anchor, _alt_anchor)

    grid                      = int(style.get("gridlines", 0))
    gridcolor, grid_linestyle = get_gridline_style(style, rng=rng)
    grid_dash                 = "dash" if grid_linestyle == "dashed" else "solid"

    show_legend = int(style.get("legend_present", 1)) == 1 and int(style.get("legend_orientation", 2)) != 0
    orient_code = int(style.get("legend_orientation", 2))

    legend_title_fs          = get_legend_title_fontsize(style, rng=rng)
    fill_color, fill_outline = get_legend_fill_and_outline(style, rng=rng)
    has_outline              = fill_outline or int(style.get("legend_outline", 0)) == 1
    plotly_coords            = legend_coords_plotly(style)

    legend_dict = dict(
        bordercolor=fg if has_outline else "rgba(0,0,0,0)",
        borderwidth=1.5 if has_outline else 0,
        bgcolor=fill_color or "rgba(0,0,0,0)",
        font=dict(color=get_legend_text_color(style, rng=rng) or fg),
        orientation="h" if orient_code in {3, 4, 8} else "v",
        **plotly_coords,
    )
    if legend_title_fs is not None:
        legend_dict["title"] = dict(
            text="Series",
            font=dict(size=legend_title_fs, color=get_legend_title_color(style, rng=rng)),
        )

    fig.update_layout(
        title=dict(text=title_text, x=x_pos, xanchor=x_anchor,
                   font=dict(color=get_title_color(style, rng=rng),
                             size=get_title_fontsize(style, rng=rng))) if title_text else None,
        plot_bgcolor=paper_bg, paper_bgcolor=paper_bg,
        font=dict(color=fg),
        xaxis=dict(
            title=context["x_label"] if orient in {0, 1, 3, 4} else None,
            color=axis_color, zeroline=False, layer="below traces",
            tickangle=0,
        ),
        yaxis=dict(
            title=context["y_label"] if orient in {0, 1, 3, 5} else None,
            color=axis_color, zeroline=False, layer="below traces",
        ),
        showlegend=show_legend,
        legend=legend_dict,
        width=760, height=480,
    )

    if context.get("x_tick_positions") and context.get("x_tick_labels"):
        positions = list(context["x_tick_positions"])
        labels    = list(context["x_tick_labels"])
        if len(positions) > 1:
            avg_chars  = max(1, sum(len(str(l)) for l in labels) / len(labels))
            max_labels = max(2, int(760 / (avg_chars * 8.5)))
            if len(positions) > max_labels:
                step      = max(1, len(positions) // max_labels)
                positions = positions[::step]
                labels    = labels[::step]
        fig.update_xaxes(tickmode="array",
                         tickvals=positions,
                         ticktext=labels)

    if grid == 0:
        fig.update_xaxes(showgrid=False, zeroline=False)
        fig.update_yaxes(showgrid=False, zeroline=False)
    elif grid == 2:
        fig.update_xaxes(showgrid=False, zeroline=False)
        fig.update_yaxes(showgrid=True, gridcolor=gridcolor, gridwidth=1, griddash=grid_dash, zeroline=False)
    else:
        fig.update_xaxes(showgrid=True, gridcolor=gridcolor, gridwidth=1, griddash=grid_dash, zeroline=False)
        fig.update_yaxes(showgrid=True, gridcolor=gridcolor, gridwidth=1, griddash=grid_dash, zeroline=False)

    axis_line_color = get_axis_color(style, rng=rng)
    outline         = int(style.get("chart_outline", 1))

    if axis_line_color is None or outline == 0:
        fig.update_xaxes(showline=False)
        fig.update_yaxes(showline=False)
    elif outline == 1:
        fig.update_xaxes(showline=True, linewidth=1, linecolor=axis_line_color, mirror=False, zeroline=False)
        fig.update_yaxes(showline=True, linewidth=1, linecolor=axis_line_color, mirror=False, zeroline=False)
    elif outline == 2:
        fig.update_xaxes(showline=True, linewidth=1, linecolor=axis_line_color, mirror=True, zeroline=False)
        fig.update_yaxes(showline=True, linewidth=1, linecolor=axis_line_color, mirror=True, zeroline=False)
    elif outline == 4:
        fig.update_xaxes(showline=True, linewidth=1, linecolor=axis_line_color, mirror=False, zeroline=False)
        fig.update_yaxes(showline=False)

    if int(style.get("image_outline", 0)) == 1:
        fig.update_layout(shapes=[dict(type="rect", xref="paper", yref="paper",
                                       x0=0, y0=0, x1=1, y1=1,
                                       line=dict(color=fg, width=1))])

    apply_tick_settings_plotly(fig, style)

    margin = dict(l=60, r=60, t=60, b=80)
    if orient_code == 1:
        margin["l"] = 180
    elif orient_code == 2:
        margin["r"] = 150
    elif orient_code in {3, 5, 6, 7, 8}:
        margin["t"] = 120
    elif orient_code == 4:
        margin["b"] = 130
    fig.update_layout(margin=margin)

    return fig

## 6. Saving and generation

In [ ]:
def new_chart_id(prefix: str = "line") -> str:
    return f"{prefix}_{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}_{uuid4().hex[:8]}"


def save_metadata(meta: dict, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2, default=str)


def ensure_output_dirs(out_root: Path) -> None:
    if CLEAR_OUTPUT and out_root.exists():
        shutil.rmtree(out_root)
    for sub in SUBDIRS:
        for lib in LIBRARIES:
            (out_root / sub / lib).mkdir(parents=True, exist_ok=True)


# def save_altair_svg(chart, out_path: Path) -> None:
#     out_path.parent.mkdir(parents=True, exist_ok=True)
#     chart.save(str(out_path), format="svg")

def install_package(pip_name: str):
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

def ensure_altair_png_support():
    try:
        import vl_convert  # noqa: F401
    except Exception:
        install_package("vl-convert-python")

def save_altair_png(chart, path: Path) -> Path:
    path = Path(path).with_suffix(".png")
    path.parent.mkdir(parents=True, exist_ok=True)
    ensure_altair_png_support()
    chart.save(str(path))
    return path


def save_plotly_png(fig, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_image(str(out_path), format="png", scale=2)


def save_matplotlib_png(fig, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        out_path,
        dpi=200,
        transparent=(
            fig.get_facecolor()[-1] == 0
            if hasattr(fig.get_facecolor(), "__len__")
            else False
        ),
    )
    plt.close(fig)


def generate_line(
    out_root: Path,
    dataset_source: str,
    library: str,
    rng_seed: int,
    param_stats: dict,
    datasets: dict | None = None,
    max_tries: int = 25,
) -> dict:
    rng      = np.random.default_rng(rng_seed)
    chart_id = new_chart_id("line")

    for attempt in range(1, max_tries + 1):
        style = harmonize_style(sample_style(rng, param_stats))

        # ── Sample data ────────────────────────────────────────────────────
        df      = None
        context = None

        if datasets:
            # Pick a random available dataset
            ds_names = list(datasets.keys())
            ds_name  = ds_names[int(rng.integers(0, len(ds_names)))]
            raw_df   = datasets[ds_name]
            spec     = DATASET_REGISTRY[ds_name]
            result   = sample_line_data_from_df(raw_df, spec, rng, style)
            if result is not None:
                df, context      = result
                dataset_source   = ds_name
            else:
                print(f"[seed {rng_seed}] Real data sampling failed → synthetic fallback")

        # Fallback to synthetic data if real data sampling failed
        if df is None:
            df, context = sample_line_data(rng=rng, style=style)

        title, subtitle = make_title(context, style=style)

        table_path = out_root / "tables"   / library / f"{chart_id}.csv"
        meta_path  = out_root / "metadata" / library / f"{chart_id}.json"
        df.to_csv(table_path, index=False)

        if library == "altair":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            chart      = render_line_altair(df, context, title, subtitle, style, rng=rng)
            save_altair_png(chart, image_path)
        elif library == "matplotlib":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig        = render_line_matplotlib(df, context, title, subtitle, style, rng=rng)
            save_matplotlib_png(fig, image_path)
        elif library == "seaborn":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig        = render_line_seaborn(df, context, title, subtitle, style, rng=rng)
            save_matplotlib_png(fig, image_path)
        elif library == "plotly":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig        = render_line_plotly(df, context, title, subtitle, style, rng=rng)
            save_plotly_png(fig, image_path)
        else:
            raise ValueError(f"Unsupported library: {library}")

        meta = {
            "chart_id":       chart_id,
            "chart_type":     "line",
            "library":        library,
            "dataset_source": dataset_source,
            "image_path":     str(image_path),
            "table_path":     str(table_path),
            "data_context":   context,
            "style":          style,
            "title":          title,
            "subtitle":       subtitle,
            "created_utc":    datetime.utcnow().isoformat() + "Z",
            "rng_seed":       rng_seed,
        }
        save_metadata(meta, meta_path)
        return meta

    raise RuntimeError(f"Failed to generate line chart after {max_tries} attempts.")


def generate_batch(
    out_root: Path,
    dataset_source: str,
    generation_plan: dict,
    param_stats: dict,
    datasets: dict | None = None,
    start_seed: int = 1000,
) -> list[dict]:
    ensure_output_dirs(out_root)
    metas = []
    seed  = start_seed
    for library, n in generation_plan.items():
        for _ in range(n):
            metas.append(generate_line(
                out_root=out_root,
                dataset_source=dataset_source,
                library=library,
                rng_seed=seed,
                param_stats=param_stats,
                datasets=datasets,
            ))
            seed += 1
    return metas

In [43]:
ensure_output_dirs(LINE_OUT_ROOT)

weights = OBSERVED_WEIGHTS

generation_plan = {
    "altair":     10,
    "matplotlib": 10,
    "seaborn":    10,
    "plotly":     10,
}

metas = generate_batch(
    out_root=LINE_OUT_ROOT,
    dataset_source="warehouse_retail",
    generation_plan=generation_plan,
    param_stats=weights,
    datasets=DATASETS,   # pass real datasets here; set to None for synthetic only
    start_seed=1000,
)

pd.DataFrame(metas)[["chart_id", "library", "dataset_source", "image_path"]].head()

,chart_id,library,dataset_source,image_path
0,line_20260614T143121_5586d5c1,altair,warehouse_retail,C:\Users\Michelle\I2R\notebooks\final_notebook...
1,line_20260614T143122_fbb24784,altair,warehouse_retail,C:\Users\Michelle\I2R\notebooks\final_notebook...
2,line_20260614T143123_2751a7ee,altair,warehouse_retail,C:\Users\Michelle\I2R\notebooks\final_notebook...
3,line_20260614T143123_2bc8bef0,altair,warehouse_retail,C:\Users\Michelle\I2R\notebooks\final_notebook...
4,line_20260614T143123_5dec8804,altair,london_borough_sector_jobs,C:\Users\Michelle\I2R\notebooks\final_notebook...


# TESTING

### Smoke test, synthetic

In [44]:
# TEST_LIBRARIES = ["matplotlib", "seaborn", "altair", "plotly"]

# base = {param: max(codes, key=codes.get) for param, codes in SAMPLING_WEIGHTS.items()}
# base.update({
#     "title_present": 1, "title_location": 1, "title_color": 0, "title_size": 0,
#     "legend_present": 1, "legend_orientation": 2, "n_lines": 3, "background": 1,
# })

# errors = []
# rng = np.random.default_rng(42)
# df, context = sample_line_data(rng, harmonize_style(base))
# title, subtitle = "Test chart", "Subtitle"

# for param_name, code_probs in SAMPLING_WEIGHTS.items():
#     for code in sorted(code_probs.keys()):
#         style = harmonize_style({**base, param_name: code})
#         for library in TEST_LIBRARIES:
#             try:
#                 if library == "matplotlib":
#                     fig = render_line_matplotlib(df, context, title, subtitle, style)
#                     plt.close(fig)
#                 elif library == "seaborn":
#                     fig = render_line_seaborn(df, context, title, subtitle, style)
#                     plt.close(fig)
#                 elif library == "altair":
#                     render_line_altair(df, context, title, subtitle, style)
#                 elif library == "plotly":
#                     render_line_plotly(df, context, title, subtitle, style)
#             except Exception as e:
#                 errors.append({
#                     "param": param_name, "code": code, "library": library,
#                     "error": f"{type(e).__name__}: {e}",
#                 })

# if errors:
#     print(f"Found {len(errors)} errors:")
#     pd.DataFrame(errors)
# else:
#     print("All parameter codes passed for all libraries.")

### Smoke test, real data

In [45]:
# TEST_LIBRARIES = ["matplotlib", "seaborn", "altair", "plotly"]

# base = {param: max(codes, key=codes.get) for param, codes in SAMPLING_WEIGHTS.items()}
# base.update({
#     "title_present": 1, "title_location": 1, "title_color": 0, "title_size": 0,
#     "legend_present": 1, "legend_orientation": 2, "n_lines": 3, "background": 1,
# })

# errors = []
# rng   = np.random.default_rng(42)
# style = harmonize_style(base)

# df_raw = DATASETS["warehouse_retail"]
# spec   = DATASET_REGISTRY["warehouse_retail"]

# print("Dataset shape:", df_raw.shape)
# print("Columns:", list(df_raw.columns))
# print("numeric_cols:", spec["numeric_cols"])
# print("group_cols:",   spec["group_cols"])

# result = sample_line_data_from_df(df_raw, spec, rng, style)
# if result is None:
#     print("Real data sampling failed — check your dataset")
# else:
#     df, context = result
#     title    = "Real data test"
#     subtitle = None

#     print(f"Sampled {len(df)} rows | series: {df['series'].unique().tolist()}")

#     for param_name, code_probs in SAMPLING_WEIGHTS.items():
#         for code in sorted(code_probs.keys()):
#             style = harmonize_style({**base, param_name: code})
#             for library in TEST_LIBRARIES:
#                 try:
#                     if library == "matplotlib":
#                         fig = render_line_matplotlib(df, context, title, subtitle, style)
#                         plt.close(fig)
#                     elif library == "seaborn":
#                         fig = render_line_seaborn(df, context, title, subtitle, style)
#                         plt.close(fig)
#                     elif library == "altair":
#                         render_line_altair(df, context, title, subtitle, style)
#                     elif library == "plotly":
#                         render_line_plotly(df, context, title, subtitle, style)
#                 except Exception as e:
#                     errors.append({
#                         "param": param_name, "code": code, "library": library,
#                         "error": f"{type(e).__name__}: {e}",
#                     })

#     if errors:
#         print(f"\nFound {len(errors)} errors:")
#         pd.DataFrame(errors)
#     else:
#         print("\nAll parameter codes passed for all libraries.")

## 7. Test run — one chart per parameter option

Generates one chart per numeric code per parameter so you can visually verify every renderer handles every code correctly.

In [46]:
# from pathlib import Path
# import re

# TEST_LIBRARIES = ["matplotlib", "seaborn", "altair", "plotly"]
# shutil.rmtree(LINE_TESTING_ROOT, ignore_errors=True)

# BASE_STYLE = {param: max(codes, key=codes.get) for param, codes in SAMPLING_WEIGHTS.items()}

# # For each parameter, define which other params must be forced
# # so the tested parameter is actually visible in the output
# PREREQS = {
#     "title_location":        {"title_present": 1},
#     "title_color":           {"title_present": 1},
#     "title_size":            {"title_present": 1},
#     "subtitle_present":      {"title_present": 1},
#     "legend_title_size":     {"legend_present": 1, "legend_orientation": 2, "n_lines": 3},
#     "legend_title_color":    {"legend_present": 1, "legend_orientation": 2,
#                               "legend_title_size": 1, "n_lines": 3},
#     "legend_text_color":     {"legend_present": 1, "legend_orientation": 2, "n_lines": 3},
#     "legend_outline":        {"legend_present": 1, "legend_orientation": 2, "n_lines": 3},
#     "legend_fill":           {"legend_present": 1, "legend_orientation": 2, "n_lines": 3},
#     "legend_orientation":    {"legend_present": 1, "n_lines": 3},
#     "label_content":         {"direct_labels": 1},
#     "label_color":           {"direct_labels": 1, "label_content": 1},
#     "gridline_color":        {"gridlines": 1},
#     "point_same_color":      {"point_shape_mode": 0},
# }

# def ensure_line_testing_dirs(out_root: Path) -> None:
#     for sub in SUBDIRS:
#         for lib in LIBRARIES:
#             (out_root / sub / lib).mkdir(parents=True, exist_ok=True)

# def sanitize_slug(text) -> str:
#     text = re.sub(r"[^a-zA-Z0-9._-]+", "_", str(text))
#     return text.strip("_").lower()

# def render_single_test_case(param_name: str, code: int, library: str, out_root: Path) -> dict:
#     style = {**BASE_STYLE}
#     style.update(PREREQS.get(param_name, {}))
#     style[param_name] = code

#     if param_name not in {"title_present", "title_location", "title_color", "title_size"}:
#         style["title_present"]  = 1
#         style["title_location"] = 1
#         style["title_color"]    = 0
#         style["title_size"]     = 0

#     style = harmonize_style(style)

#     rng = np.random.default_rng(2026)
#     df, context = sample_line_data(rng, style)
#     title    = f"{param_name.replace('_', ' ')} = {code}"
#     subtitle = None
#     chart_id = sanitize_slug(f"{param_name}__code_{code}")

#     table_path = out_root / "tables"   / library / f"{chart_id}.csv"
#     meta_path  = out_root / "metadata" / library / f"{chart_id}.json"
#     df.to_csv(table_path, index=False)

#     if library == "altair":
#         image_path = out_root / "images" / library / f"{chart_id}.svg"
#         chart      = render_line_altair(df, context, title, subtitle, style)
#         save_altair_svg(chart, image_path)
#     elif library == "matplotlib":
#         image_path = out_root / "images" / library / f"{chart_id}.png"
#         fig        = render_line_matplotlib(df, context, title, subtitle, style)
#         save_matplotlib_png(fig, image_path)
#     elif library == "seaborn":
#         image_path = out_root / "images" / library / f"{chart_id}.png"
#         fig        = render_line_seaborn(df, context, title, subtitle, style)
#         save_matplotlib_png(fig, image_path)
#     else:
#         image_path = out_root / "images" / library / f"{chart_id}.png"
#         fig        = render_line_plotly(df, context, title, subtitle, style)
#         save_plotly_png(fig, image_path)

#     meta = {
#         "chart_id": chart_id, "library": library,
#         "param_tested": param_name, "code_tested": code,
#         "style": style,
#         "image_path": str(image_path), "table_path": str(table_path),
#     }
#     save_metadata(meta, meta_path)
#     return meta


# ensure_line_testing_dirs(LINE_TESTING_ROOT)
# all_test_metas = []

# for param_name, code_probs in SAMPLING_WEIGHTS.items():
#     for code in sorted(code_probs.keys()):
#         for library in TEST_LIBRARIES:
#             all_test_metas.append(
#                 render_single_test_case(param_name, code, library, LINE_TESTING_ROOT)
#             )

# pd.DataFrame(all_test_metas)[[
#     "chart_id", "library", "param_tested", "code_tested", "image_path"
# ]].head(20)